# Ingesta completa distribuida: Aeropuertos de Colombia + IEM METAR histórico + datos.gov.co

En este notebook se construye el flujo de **ingesta batch** del proyecto usando **PySpark DataFrames** en Databricks.

La extracción desde APIs se mantiene controlada desde el driver por límites de tasa y reintentos.  
La transformación analítica **Bronze → Silver → Gold** se ejecuta con Spark DataFrames.

## Secciones
0. INSTALACION OPCIONAL DE DEPENDENCIAS
1. IMPORTS
2. CONFIGURACIÓN GENERAL
3. URLS DE LAS FUENTES
4. FUNCIONES AUXILIARES GENERALES
5. API SOCRATA Y DATOS.GOV.CO
6. DESCARGA Y DIMENSIÓN DE AEROPUERTOS COLOMBIANOS
7. DESCARGA COMPLETA DE DATOS.GOV.CO
8. DESCARGA DE FILAS DE LOS DOS DATASETS
9. FUNCIONES DE TRANSFORMACIÓN PARA DATOS.GOV.CO
10. CONSTRUIR OPERACIONES Y ORIGEN-DESTINO A NIVEL AEROPUERTO-MES
12. DESCARGA IEM HISTÓRICO 2020-01 A 2025-12
13. NORMALIZAR IEM Y AGREGAR A NIVEL MENSUAL
14. CONSTRUIR DATASET GOLD FINAL
15. CREAR TARGET BAJO / MEDIO / ALTO
16. DIAGNÓSTICOS DE COBERTURA
17. INSPECCIÓN FINAL DE ARCHIVOS GENERADOS

## Decisiones técnicas aplicadas

1. Solo se usan aeropuertos de Colombia.
2. Solo se aceptan aeropuertos con `icao_code` válido tipo `SKXX`.
3. No se descarga AviationWeather ni TAF.
4. La fuente meteorológica base es IEM ASOS/METAR histórico.
5. El rango temporal del modelo es `2020-01` a `2025-12`.
6. La unidad final de análisis es `icao_code + fecha_mes`.
7. El dataset de operaciones construye el target.
8. El dataset origen-destino construye variables explicativas de salidas, llegadas, pasajeros, carga y conectividad.
9. Las transformaciones principales se ejecutan con Spark.
10. Para conservar compatibilidad con el volumen original, los Parquet finales se dejan como archivo único en la misma ruta `.parquet`.

### 0. INSTALACIÓN OPCIONAL DE DEPENDENCIAS

In [1]:
# Databricks normalmente ya incluye requests y PySpark.
# Descomenta solo si tu cluster lo requiere.
# %pip install -q requests
%pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 8.9 MB/s  0:00:400:00:0100:02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-4.1.2-py2.py3-none-any.whl size=456079515 sha256=eaea1fdd2d069513b2219fc72caeed92e82872abb226735895aa89536abace68
  Stored in directory: /home/madacohe/.cache/pip/wheels/e6/9c/35/b08622081a09dc48b9467b570ae170519430915aa3c8d27cf9
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]
Note: you may need to restart the kernel to use updated packages.


### 1. IMPORTS

In [1]:
import os
import json
import math
import re
import time
import unicodedata
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from io import StringIO
import csv

import requests
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("ProyectoAeropuertosLocal")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 50)

print("Imports OK")
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/28 22:25:53 WARN Utils: Your hostname, madacohe-X532FLC-S532FL, resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface wlo1)
26/05/28 22:25:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/28 22:25:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Imports OK
Spark version: 4.1.2


### 2. CONFIGURACIÓN GENERAL

#### 2.1. Paths principales.

In [2]:
BASE_DIR = Path("data_proyecto_aeropuertos")

BRONZE_DIR = BASE_DIR / "bronze"
SILVER_DIR = BASE_DIR / "silver"
GOLD_DIR = BASE_DIR / "gold"

#### 2.2. Paths Bronze.

In [3]:
BRONZE_AIRPORTS_MASTER_DIR = BRONZE_DIR / "airports" / "master"
BRONZE_IEM_BY_AIRPORT_DIR = BRONZE_DIR / "iem" / "by_airport"

BRONZE_TRAFICO_OD_PAGES_DIR = BRONZE_DIR / "trafico_origen_destino" / "pages"
BRONZE_OPERACIONES_PAGES_DIR = BRONZE_DIR / "operaciones_aereas" / "pages"

BRONZE_METADATA_DIR = BRONZE_DIR / "metadata"

#### 2.3. Paths SILVER.

In [4]:
SILVER_AIRPORTS_DIR = SILVER_DIR / "airports"

SILVER_IEM_DIR = SILVER_DIR / "iem"
SILVER_IEM_BY_AIRPORT_DIR = SILVER_IEM_DIR / "by_airport"

SILVER_TRAFICO_OD_DIR = SILVER_DIR / "trafico_origen_destino"
SILVER_OPERACIONES_DIR = SILVER_DIR / "operaciones_aereas"

SILVER_DIAGNOSTICS_DIR = SILVER_DIR / "diagnostics"

#### 2.4. Paths GOLD.

In [5]:
GOLD_MONTHLY_DIR = GOLD_DIR / "monthly_airport_dataset"

#### 2.5. Rangos temporales del modelo.

In [6]:
MODEL_START_MONTH = "2020-01-01"
MODEL_END_MONTH = "2025-12-01"

# Para IEM se usa fin exclusivo para incluir todo diciembre 2025.
IEM_START = "2020-01-01T00:00:00Z"
IEM_END_EXCLUSIVE = "2026-02-01T00:00:00Z"

#### 2.6. API y Descargas.

In [7]:
DATOS_GOV_LIMIT = 50000
DATOS_GOV_APP_TOKEN = None  # Token opcional de datos.gov.co.

OVERWRITE_BRONZE = True

RUN_IEM_DOWNLOAD = True
MAX_IEM_AIRPORTS = None  # None = todos. Para prueba rápida usar 5.

#### 2.7. Límites de tasa por fuente.

In [8]:
RATE_LIMITS = {
    "ourairports": {
        "min_interval_seconds": 0.5,
        "max_retries": 4,
        "backoff_base_seconds": 2.0,
    },
    "iem": {
        "min_interval_seconds": 2.0,
        "max_retries": 8,
        "backoff_base_seconds": 4.0,
    },
    "socrata": {
        "min_interval_seconds": 1.0,
        "max_retries": 6,
        "backoff_base_seconds": 3.0,
    },
}

LAST_REQUEST_TS: dict[str, float] = {}

#### 2.8. Crear carpetas para descargas.

In [9]:
for folder in [
    BRONZE_AIRPORTS_MASTER_DIR,
    BRONZE_IEM_BY_AIRPORT_DIR,
    BRONZE_TRAFICO_OD_PAGES_DIR,
    BRONZE_OPERACIONES_PAGES_DIR,
    BRONZE_METADATA_DIR,
    SILVER_AIRPORTS_DIR,
    SILVER_IEM_DIR,
    SILVER_IEM_BY_AIRPORT_DIR,
    SILVER_TRAFICO_OD_DIR,
    SILVER_OPERACIONES_DIR,
    SILVER_DIAGNOSTICS_DIR,
    GOLD_MONTHLY_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Estructura de carpetas creada correctamente.")

Estructura de carpetas creada correctamente.


### 3. URLS DE LAS FUENTES

In [10]:
OURAIRPORTS_CO_URL = "https://ourairports.com/countries/CO/airports.csv"

IEM_ASOS_URL = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

DATOS_GOV_BASE = "https://www.datos.gov.co/resource"
DATOS_GOV_METADATA_BASE = "https://www.datos.gov.co/api/views"

TRAFICO_OD_RESOURCE = "gb6w-ynu4"
OPERACIONES_RESOURCE = "jh8x-n6h6"

#### 3.1. Sesión HTTP

In [11]:
session = requests.Session()
session.headers.update({
    "User-Agent": "ProyectoIntegradorAeropuertosColombia/1.0"
})

#### 3.2. Códigos excluidos.

In [12]:
CODIGOS_SKXX_EXCLUIDOS = {"SKXX"}
DELETE_EMPTY_SILVER_IEM_AIRPORT_DIRS = True

### 4. FUNCIONES AUXILIARES GENERALES

In [13]:
def ensure_parent_dir(output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)


def merge_headers(custom_headers: dict | None = None, app_token: str | None = None) -> dict:
    headers = {"User-Agent": "ProyectoIntegradorAeropuertosColombia/1.0"}

    if app_token:
        headers["X-App-Token"] = app_token

    if custom_headers:
        headers.update(custom_headers)

    return headers


def wait_for_service_rate_limit(service_name: str, min_interval_seconds: float) -> None:
    now_ts = time.monotonic()
    last_ts = LAST_REQUEST_TS.get(service_name)

    if last_ts is None:
        return

    elapsed = now_ts - last_ts
    remaining = min_interval_seconds - elapsed

    if remaining > 0:
        time.sleep(remaining)

In [14]:
def request_with_retry(
    service_name: str,
    method: str,
    url: str,
    params: dict | None = None,
    headers: dict | None = None,
    app_token: str | None = None,
    timeout: int = 180,
    stream: bool = False,
    min_interval_seconds: float | None = None,
    max_retries: int | None = None,
    backoff_base_seconds: float | None = None,
) -> requests.Response:
    service_cfg = RATE_LIMITS.get(service_name, {})

    min_interval_seconds = (
        min_interval_seconds
        if min_interval_seconds is not None
        else service_cfg.get("min_interval_seconds", 1.5)
    )

    max_retries = (
        max_retries
        if max_retries is not None
        else service_cfg.get("max_retries", 6)
    )

    backoff_base_seconds = (
        backoff_base_seconds
        if backoff_base_seconds is not None
        else service_cfg.get("backoff_base_seconds", 3.0)
    )

    final_headers = merge_headers(headers, app_token=app_token)
    last_exception = None

    for attempt in range(max_retries):
        wait_for_service_rate_limit(service_name, min_interval_seconds)

        try:
            response = session.request(
                method=method,
                url=url,
                params=params,
                headers=final_headers,
                timeout=timeout,
                stream=stream,
            )

            LAST_REQUEST_TS[service_name] = time.monotonic()

            if response.status_code in (429, 503, 504):
                retry_after_header = response.headers.get("Retry-After")

                if retry_after_header and retry_after_header.isdigit():
                    wait_seconds = float(retry_after_header)
                else:
                    wait_seconds = backoff_base_seconds * (2 ** attempt)

                print(
                    f"[WARN] {service_name} respondió HTTP {response.status_code}. "
                    f"Reintentando en {wait_seconds:.1f} s "
                    f"({attempt + 1}/{max_retries})..."
                )

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            return response

        except (
            requests.exceptions.ConnectionError,
            requests.exceptions.Timeout,
            requests.exceptions.ChunkedEncodingError,
            requests.exceptions.RequestException,
        ) as exc:
            LAST_REQUEST_TS[service_name] = time.monotonic()
            last_exception = exc

            wait_seconds = backoff_base_seconds * (2 ** attempt)

            print(
                f"[WARN] {service_name} lanzó {type(exc).__name__}: {exc}. "
                f"Reintentando en {wait_seconds:.1f} s "
                f"({attempt + 1}/{max_retries})..."
            )

            time.sleep(wait_seconds)
            continue

    raise RuntimeError(
        f"No se pudo completar la solicitud a {service_name} "
        f"después de {max_retries} intentos. Último error: {last_exception}"
    )

#### 4.1. Lectura y escritura Parquet con Spark en formato distribuido

In [15]:
def remove_path_if_exists(path: Path) -> None:
    """
    Elimina un archivo o carpeta si ya existe.

    Esto permite reemplazar salidas antiguas tipo archivo único
    por salidas Spark tipo carpeta Parquet distribuida.
    """
    path = Path(path)

    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()

        print(f"[OK] Path anterior eliminado: {path}")


def write_spark_parquet_dataset(
    df: DataFrame,
    output_path: Path,
    mode: str = "overwrite",
    partition_by: list[str] | None = None,
) -> None:
    """
    Escribe un Spark DataFrame como dataset Parquet distribuido.

    Spark guarda Parquet como carpeta, por ejemplo:

        archivo.parquet/
        ├── part-00000-....snappy.parquet
        ├── part-00001-....snappy.parquet
        └── _SUCCESS

    Esta función NO usa coalesce(1), por lo tanto conserva la escritura
    distribuida propia de Spark.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    remove_path_if_exists(output_path)

    writer = (
        df.write
        .mode(mode)
        .option("compression", "snappy")
    )

    if partition_by:
        writer = writer.partitionBy(*partition_by)

    writer.parquet(str(output_path))

    n_rows = df.count()
    n_cols = len(df.columns)

    part_files = sorted(output_path.glob("part-*.parquet"))
    n_part_files = len(part_files)

    print(
        f"[OK] Dataset Parquet Spark guardado: {output_path} | "
        f"filas={n_rows:,} | columnas={n_cols:,} | "
        f"part-files={n_part_files:,}"
    )


def read_spark_parquet(path_or_paths) -> DataFrame:
    """
    Lee Parquet usando Spark.

    Soporta:
    - Un solo path: Path o string.
    - Varios paths: list, tuple o set de Path/string.

    Esto evita que Spark reciba accidentalmente una lista convertida a string,
    lo cual causa errores tipo:
    Illegal file pattern: Unclosed character class at pos 2: `[`
    """

    if isinstance(path_or_paths, (list, tuple, set)):
        paths = [
            str(path)
            for path in path_or_paths
        ]

        if not paths:
            raise ValueError("La lista de paths Parquet está vacía.")

        return spark.read.parquet(*paths)

    return spark.read.parquet(str(path_or_paths))


def save_json_to_file(data: list | dict, output_path: Path) -> None:
    """
    Guarda un objeto Python como archivo JSON.
    """
    ensure_parent_dir(output_path)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[OK] JSON guardado: {output_path}")

#### 4.2. Utilidades de texto, columnas y tipos.

In [16]:
def normalize_text_key(value: Any) -> str:
    if value is None:
        return ""

    text = str(value)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")

    return text


def find_column_spark(
    df: DataFrame,
    candidates: list[str],
    required: bool = True,
    label: str = "",
) -> str | None:
    normalized_to_original = {
        normalize_text_key(col): col
        for col in df.columns
    }

    for candidate in candidates:
        key = normalize_text_key(candidate)

        if key in normalized_to_original:
            return normalized_to_original[key]

    if required:
        raise ValueError(
            f"No se encontró la columna requerida {label or candidates}. "
            f"Candidatas: {candidates}. "
            f"Columnas disponibles: {df.columns}"
        )

    return None


def normalizar_codigo_expr(col_name: str):
    raw = F.upper(F.trim(F.col(col_name).cast("string")))

    return (
        F.when(raw.isNull(), F.lit(None).cast("string"))
        .when(raw.isin("", "NAN", "NONE", "NULL", "NA", "N/A"), F.lit(None).cast("string"))
        .otherwise(raw)
    )


def es_codigo_skxx_valido_expr(col_expr):
    return (
        col_expr.rlike(r"^SK[A-Z]{2}$")
        & (~col_expr.isin(list(CODIGOS_SKXX_EXCLUIDOS)))
    )


def to_numeric_expr(col_name: str):
    cleaned = F.regexp_replace(F.trim(F.col(col_name).cast("string")), ",", "")
    return cleaned.cast("double")


def safe_divide_expr(numerator_col: str, denominator_col: str):
    numerator = F.col(numerator_col).cast("double")
    denominator = F.col(denominator_col).cast("double")

    return (
        F.when(denominator.isNull() | (denominator == 0), F.lit(0.0))
        .otherwise(numerator / denominator)
    )


def crear_fecha_mes_desde_anio_mes_spark(df: DataFrame, anio_col: str, mes_col: str) -> DataFrame:
    anio = F.regexp_extract(F.col(anio_col).cast("string"), r"(\d{4})", 1)
    mes = F.regexp_extract(F.col(mes_col).cast("string"), r"(\d{1,2})", 1)

    return df.withColumn(
        "fecha_mes",
        F.to_date(
            F.concat_ws(
                "-",
                anio,
                F.lpad(mes, 2, "0"),
                F.lit("01"),
            ),
            "yyyy-MM-dd",
        )
    )

#### 4.3. Descarga CSV como Parquet Spark.

In [17]:
def download_csv_as_spark_parquet(
    service_name: str,
    url: str,
    output_path: Path,
    params: dict | None = None,
    headers: dict | None = None,
    overwrite: bool = False,
    spark_read_options: dict | None = None,
) -> DataFrame:
    """
    Descarga una respuesta CSV por HTTP, la convierte a Spark DataFrame
    y la guarda como Parquet.

    Esta versión NO usa spark.read.csv sobre archivo temporal, porque en Databricks
    puede leer 0 filas en algunos casos. En su lugar, usa csv.reader y luego
    spark.createDataFrame.
    """

    output_path = Path(output_path)

    if output_path.suffix.lower() != ".parquet":
        raise ValueError("output_path debe terminar en .parquet")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        print(f"[SKIP] Ya existe Parquet: {output_path}")
        return read_spark_parquet(output_path)

    response = request_with_retry(
        service_name=service_name,
        method="GET",
        url=url,
        params=params,
        headers=headers,
        timeout=300,
        stream=False,
    )

    text = response.text
    text_head = text[:500].lower()

    if not text.strip():
        raise ValueError(f"La respuesta de {service_name} está vacía.")

    if "<html" in text_head or "<!doctype" in text_head:
        raise ValueError(f"La respuesta de {service_name} parece HTML, no CSV.")

    if "service unavailable" in text_head:
        raise ValueError(f"La respuesta de {service_name} parece error del servidor, no CSV.")

    # ------------------------------------------------------------
    # 1. Leer CSV en memoria con parser estándar de Python.
    # ------------------------------------------------------------
    csv_rows = list(csv.reader(StringIO(text)))

    csv_rows = [
        row for row in csv_rows
        if row and any(str(value).strip() for value in row)
    ]

    if not csv_rows:
        raise ValueError(f"No se encontraron filas CSV válidas para {service_name}.")

    header = csv_rows[0]
    data_rows_raw = csv_rows[1:]

    # ------------------------------------------------------------
    # 2. Limpiar encabezados para Spark.
    # ------------------------------------------------------------
    clean_columns = []
    seen = {}

    for idx, col_name in enumerate(header):
        col_clean = str(col_name).strip()

        if not col_clean:
            col_clean = f"col_{idx}"

        if col_clean in seen:
            seen[col_clean] += 1
            col_clean = f"{col_clean}_{seen[col_clean]}"
        else:
            seen[col_clean] = 0

        clean_columns.append(col_clean)

    n_cols = len(clean_columns)

    # ------------------------------------------------------------
    # 3. Ajustar filas al número de columnas.
    #    Si una fila viene corta, se rellena.
    #    Si viene larga, se recorta.
    # ------------------------------------------------------------
    data_rows = []

    for row in data_rows_raw:
        row_fixed = list(row)

        if len(row_fixed) < n_cols:
            row_fixed = row_fixed + [None] * (n_cols - len(row_fixed))

        if len(row_fixed) > n_cols:
            row_fixed = row_fixed[:n_cols]

        data_rows.append(tuple(row_fixed))

    if len(data_rows) == 0 and service_name != "iem":
        raise ValueError(
            f"El CSV de {service_name} tiene encabezado pero 0 filas de datos. "
            f"Primeros 500 caracteres: {text[:500]}"
        )

    schema = T.StructType([
        T.StructField(col_name, T.StringType(), True)
        for col_name in clean_columns
    ])

    df = spark.createDataFrame(data_rows, schema=schema)

    n_rows = df.count()

    if n_rows == 0 and service_name != "iem":
        raise ValueError(
            f"Spark creó 0 filas para {service_name}. No se guardará Parquet vacío. "
            f"Primeros 500 caracteres: {text[:500]}"
        )

    write_spark_parquet_dataset(df, output_path)

    print(
        f"[OK] CSV descargado y convertido a Parquet Spark: {output_path} | "
        f"filas={n_rows:,} | columnas={len(clean_columns):,}"
    )

    return read_spark_parquet(output_path)

### 5. API SOCRATA y DATOS.GOV.CO

In [18]:
def get_dataset_metadata(
    resource_id: str,
    app_token: str | None = None,
) -> dict:
    url = f"{DATOS_GOV_METADATA_BASE}/{resource_id}.json"

    response = request_with_retry(
        service_name="socrata",
        method="GET",
        url=url,
        app_token=app_token,
        timeout=120,
        stream=False,
    )

    return response.json()


def metadata_columns_to_spark_df(metadata: dict) -> DataFrame:
    rows = []

    for col in metadata.get("columns", []):
        rows.append({
            "name": col.get("name"),
            "fieldName": col.get("fieldName"),
            "dataTypeName": col.get("dataTypeName"),
            "description": col.get("description"),
        })

    return spark.createDataFrame(rows)


def apply_metadata_display_names_spark(df: DataFrame, metadata: dict) -> DataFrame:
    df_out = df

    for col in metadata.get("columns", []):
        field_name = col.get("fieldName")
        display_name = col.get("name")

        if field_name in df_out.columns and display_name and field_name != display_name:
            df_out = df_out.withColumnRenamed(field_name, display_name)

    return df_out

In [19]:
def get_dataset_row_count(
    resource_id: str,
    app_token: str | None = None,
    where: str | None = None,
) -> int:
    url = f"{DATOS_GOV_BASE}/{resource_id}.json"

    params = {"$select": "count(*)"}

    if where:
        params["$where"] = where

    response = request_with_retry(
        service_name="socrata",
        method="GET",
        url=url,
        params=params,
        app_token=app_token,
        timeout=120,
        stream=False,
    )

    data = response.json()

    return int(data[0]["count"])


def download_socrata_dataset_all_rows_to_json_pages(
    resource_id: str,
    output_dir: Path,
    base_name: str,
    limit: int = DATOS_GOV_LIMIT,
    app_token: str | None = DATOS_GOV_APP_TOKEN,
    where: str | None = None,
    order_by: str | None = None,
    overwrite: bool = OVERWRITE_BRONZE,
) -> DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)

    existing_pages = sorted(output_dir.glob(f"{base_name}_page_*.json"))

    if existing_pages and not overwrite:
        print(f"[SKIP] Ya existen páginas para {base_name}. Cargando con Spark desde disco...")
        return spark.read.option("multiLine", "true").json([str(p) for p in existing_pages])

    total_rows = get_dataset_row_count(
        resource_id=resource_id,
        app_token=app_token,
        where=where,
    )

    print(f"[INFO] Total filas esperadas para {resource_id} ({base_name}): {total_rows:,}")

    offset = 0
    page = 1

    while offset < total_rows:
        url = f"{DATOS_GOV_BASE}/{resource_id}.json"

        params = {
            "$limit": limit,
            "$offset": offset,
        }

        if where:
            params["$where"] = where

        if order_by:
            params["$order"] = order_by

        print(
            f"[INFO] Descargando {base_name} | "
            f"página {page} | offset {offset:,} | limit {limit:,}"
        )

        response = request_with_retry(
            service_name="socrata",
            method="GET",
            url=url,
            params=params,
            app_token=app_token,
            timeout=180,
            stream=False,
        )

        records = response.json()

        output_path = output_dir / f"{base_name}_page_{page:04d}.json"
        save_json_to_file(records, output_path)

        if not records:
            print("[INFO] Página vacía recibida. Fin de descarga.")
            break

        offset += limit
        page += 1

    json_pages = sorted(output_dir.glob(f"{base_name}_page_*.json"))

    if not json_pages:
        return spark.createDataFrame([], T.StructType([]))

    df = spark.read.option("multiLine", "true").json([str(p) for p in json_pages])

    print(f"[OK] Descarga finalizada para {base_name}. Filas leídas por Spark: {df.count():,}")

    return df

### 6. DESCARGA Y DIMENSIÓN DE AEROPUERTOS COLOMBIANOS

In [20]:
def get_airport_paths(icao: str) -> dict[str, Path]:
    icao = icao.strip().upper()

    if not icao:
        raise ValueError("El código ICAO no puede estar vacío.")

    return {
        "bronze_iem_metar_dir": BRONZE_IEM_BY_AIRPORT_DIR / icao / "metar",
        "silver_iem_metar_dir": SILVER_IEM_BY_AIRPORT_DIR / icao / "metar",
    }


def create_airport_folder_structure(icao_codes: list[str]) -> None:
    for icao in icao_codes:
        icao = str(icao).strip().upper()

        if not icao:
            continue

        paths = get_airport_paths(icao)

        for _, folder in paths.items():
            folder.mkdir(parents=True, exist_ok=True)

    print("[OK] Estructura por aeropuerto creada.")

In [21]:
CODE_TO_ICAO_OVERRIDES = {
    "MCJ": "SKLM",
}


def construir_dim_airports_colombia_skxx_spark(
    master_parquet_path: Path,
    supplemental_icao_codes: list[str] | None = None,
) -> tuple[DataFrame, list[str], dict, set, DataFrame]:
    df = read_spark_parquet(master_parquet_path)

    if "iso_country" not in df.columns:
        raise ValueError("Falta columna obligatoria en OurAirports: iso_country")

    df_co = (
        df
        .withColumn("iso_country_clean", normalizar_codigo_expr("iso_country"))
        .filter(F.col("iso_country_clean") == "CO")
    )

    candidate_cols = [
        col for col in ["icao_code", "gps_code", "ident", "local_code"]
        if col in df_co.columns
    ]

    final_expr = None
    source_expr = None

    for col in candidate_cols:
        clean_col = normalizar_codigo_expr(col)
        valid_expr = es_codigo_skxx_valido_expr(clean_col)

        final_expr = (
            F.when(valid_expr, clean_col)
            if final_expr is None
            else final_expr.when(valid_expr, clean_col)
        )

        source_expr = (
            F.when(valid_expr, F.lit(col))
            if source_expr is None
            else source_expr.when(valid_expr, F.lit(col))
        )

    if final_expr is None:
        final_expr = F.lit(None).cast("string")
        source_expr = F.lit(None).cast("string")
    else:
        final_expr = final_expr.otherwise(F.lit(None).cast("string"))
        source_expr = source_expr.otherwise(F.lit(None).cast("string"))

    df_co = (
        df_co
        .withColumn("icao_code_final", final_expr)
        .withColumn("icao_code_source_col", source_expr)
    )

    df_validos = df_co.filter(F.col("icao_code_final").isNotNull())
    df_descartados = df_co.filter(F.col("icao_code_final").isNull())

    df_validos = df_validos.dropDuplicates(["icao_code_final"])

    dim_candidate_cols = [
        "id",
        "ident",
        "icao_code",
        "gps_code",
        "local_code",
        "icao_code_final",
        "icao_code_source_col",
        "name",
        "type",
        "latitude_deg",
        "longitude_deg",
        "elevation_ft",
        "iso_country",
        "iso_region",
        "region_name",
        "municipality",
        "scheduled_service",
        "iata_code",
    ]

    dim_cols = [col for col in dim_candidate_cols if col in df_validos.columns]

    dim_airports_colombia = df_validos.select(*dim_cols)

    if "icao_code" in dim_airports_colombia.columns:
        dim_airports_colombia = dim_airports_colombia.withColumnRenamed(
            "icao_code",
            "icao_code_original",
        )

    dim_airports_colombia = dim_airports_colombia.withColumnRenamed(
        "icao_code_final",
        "icao_code",
    )

    if supplemental_icao_codes is not None:
        existentes = {
            row["icao_code"]
            for row in dim_airports_colombia.select("icao_code").distinct().collect()
        }

        nuevos = sorted({
            str(code).strip().upper()
            for code in supplemental_icao_codes
            if isinstance(code, str)
            and re.match(r"^SK[A-Z]{2}$", str(code).strip().upper())
            and str(code).strip().upper() not in existentes
            and str(code).strip().upper() not in CODIGOS_SKXX_EXCLUIDOS
        })

        if nuevos:
            schema = dim_airports_colombia.schema
            rows = []

            for code in nuevos:
                base = {field.name: None for field in schema.fields}
                base.update({
                    "ident": code,
                    "icao_code_original": None,
                    "gps_code": code,
                    "local_code": None,
                    "icao_code": code,
                    "icao_code_source_col": "supplemental_from_operations",
                    "type": "unknown_from_operations",
                    "iso_country": "CO",
                })
                rows.append(base)

            dim_airports_colombia = dim_airports_colombia.unionByName(
                spark.createDataFrame(rows, schema=schema),
                allowMissingColumns=True,
            )

            print(f"[INFO] Códigos suplementarios agregados: {len(nuevos)}")

    dim_airports_colombia = (
        dim_airports_colombia
        .dropDuplicates(["icao_code"])
        .orderBy("icao_code")
    )

    icao_codes = [
        row["icao_code"]
        for row in dim_airports_colombia.select("icao_code").distinct().orderBy("icao_code").collect()
    ]

    icaos_colombia_set = set(icao_codes)

    columnas_codigo = [
        col for col in [
            "icao_code",
            "ident",
            "icao_code_original",
            "gps_code",
            "local_code",
            "iata_code",
        ]
        if col in dim_airports_colombia.columns
    ]

    code_to_icao = {}
    conflictos = []
    conflictos_resueltos = []

    select_cols_codigo = list(dict.fromkeys(["icao_code"] + columnas_codigo))
    rows = dim_airports_colombia.select(*select_cols_codigo).collect()

    for row in rows:
        icao = row["icao_code"]

        if icao is None:
            continue

        icao = str(icao).strip().upper()

        for col in columnas_codigo:
            codigo = row[col]

            if codigo is None:
                continue

            codigo = str(codigo).strip().upper()

            if codigo in ["", "NAN", "NONE", "NULL", "NA", "N/A"]:
                continue

            if codigo in CODE_TO_ICAO_OVERRIDES:
                icao_preferido = CODE_TO_ICAO_OVERRIDES[codigo]

                if icao != icao_preferido:
                    conflictos_resueltos.append({
                        "codigo": codigo,
                        "icao_preferido": icao_preferido,
                        "icao_ignorado": icao,
                        "columna": col,
                        "criterio": "override_manual",
                    })
                    continue

                code_to_icao[codigo] = icao_preferido
                continue

            if codigo not in code_to_icao:
                code_to_icao[codigo] = icao
                continue

            if code_to_icao[codigo] == icao:
                continue

            conflictos.append({
                "codigo": codigo,
                "icao_existente": code_to_icao[codigo],
                "icao_nuevo_ignorado": icao,
                "columna": col,
                "criterio": "conservar_existente",
            })

    for codigo, icao_preferido in CODE_TO_ICAO_OVERRIDES.items():
        code_to_icao[codigo] = icao_preferido

    print(f"[OK] Aeropuertos colombianos SKXX válidos: {len(icao_codes):,}")
    print(f"[OK] Códigos alternativos mapeados a ICAO colombiano: {len(code_to_icao):,}")
    print(f"[INFO] Aeropuertos colombianos descartados por no tener SKXX válido: {df_descartados.count():,}")
    print(f"[INFO] Conflictos no resueltos de mapeo detectados: {len(conflictos):,}")
    print(f"[INFO] Conflictos resueltos por override manual: {len(conflictos_resueltos):,}")

    if conflictos:
        write_spark_parquet_dataset(
            spark.createDataFrame(conflictos),
            SILVER_DIAGNOSTICS_DIR / "airports_code_to_icao_conflictos_no_resueltos.parquet",
        )

    if conflictos_resueltos:
        write_spark_parquet_dataset(
            spark.createDataFrame(conflictos_resueltos),
            SILVER_DIAGNOSTICS_DIR / "airports_code_to_icao_conflictos_resueltos.parquet",
        )

    return dim_airports_colombia, icao_codes, code_to_icao, icaos_colombia_set, df_descartados

In [22]:
airports_master_path = BRONZE_AIRPORTS_MASTER_DIR / "airports_colombia.parquet"

df_airports_bronze = download_csv_as_spark_parquet(
    service_name="ourairports",
    url=OURAIRPORTS_CO_URL,
    output_path=airports_master_path,
    overwrite=OVERWRITE_BRONZE,
)

dim_airports, ICAO_CODES, code_to_icao, icaos_colombia_set, airports_descartados_sin_skxx = construir_dim_airports_colombia_skxx_spark(
    master_parquet_path=airports_master_path,
    supplemental_icao_codes=None,
)

[OK] Path anterior eliminado: data_proyecto_aeropuertos/bronze/airports/master/airports_colombia.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/airports/master/airports_colombia.parquet | filas=736 | columnas=24 | part-files=8
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/airports/master/airports_colombia.parquet | filas=736 | columnas=24


[OK] Aeropuertos colombianos SKXX válidos: 146
[OK] Códigos alternativos mapeados a ICAO colombiano: 304
[INFO] Aeropuertos colombianos descartados por no tener SKXX válido: 590
[INFO] Conflictos no resueltos de mapeo detectados: 0
[INFO] Conflictos resueltos por override manual: 1
[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/diagnostics/airports_code_to_icao_conflictos_resueltos.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/airports_code_to_icao_conflictos_resueltos.parquet | filas=1 | columnas=5 | part-files=2


In [23]:
create_airport_folder_structure(ICAO_CODES)

write_spark_parquet_dataset(
    dim_airports,
    SILVER_AIRPORTS_DIR / "dim_airports_colombia_skxx.parquet",
)

write_spark_parquet_dataset(
    airports_descartados_sin_skxx,
    SILVER_DIAGNOSTICS_DIR / "airports_colombia_descartados_sin_skxx.parquet",
)

save_json_to_file(
    code_to_icao,
    SILVER_AIRPORTS_DIR / "code_to_icao_colombia_skxx.json",
)

[OK] Estructura por aeropuerto creada.
[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/airports/dim_airports_colombia_skxx.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/airports/dim_airports_colombia_skxx.parquet | filas=146 | columnas=18 | part-files=1
[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/diagnostics/airports_colombia_descartados_sin_skxx.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/airports_colombia_descartados_sin_skxx.parquet | filas=590 | columnas=27 | part-files=8
[OK] JSON guardado: data_proyecto_aeropuertos/silver/airports/code_to_icao_colombia_skxx.json


In [24]:
print("Primeros aeropuertos válidos:")
display(dim_airports.limit(10))

Primeros aeropuertos válidos:


id,ident,icao_code_original,gps_code,local_code,icao_code,icao_code_source_col,name,type,latitude_deg,longitude_deg,elevation_ft,iso_country,iso_region,region_name,municipality,scheduled_service,iata_code
40634,SK-054,,SKAA,PAZ,SKAA,gps_code,Caño Garza Airport,small_airport,5.591667,-71.589444,544,CO,CO-CAS,Casanare Department,Paz de Ariporo,0,
30615,SKAC,SKAC,SKAC,ACR,SKAC,icao_code,Araracuara Airport,small_airport,-0.600854,-72.398011,1250,CO,CO-CAQ,Caquetá Department,Araracuara,0,ACR
30613,SKAD,SKAD,SKAD,ACD,SKAD,icao_code,Alcides Fernández...,small_airport,8.497847,-77.274106,50,CO,CO-CHO,Chocó Department,Acandí,0,ACD
32306,SKAG,SKAG,SKAG,AGH,SKAG,icao_code,Hacaritama Airport,small_airport,8.247,-73.5814,545,CO,CO-CES,César Department,Aguachica,0,HAY
40796,SK-149,,SKAL,LLO,SKAL,gps_code,Calenturitas Airport,small_airport,9.652073,-73.495134,195,CO,CO-CES,César Department,La Loma,0,
30620,SKAM,SKAM,SKAM,AFI,SKAM,icao_code,Amalfi Airport,small_airport,6.895033,-75.047334,5507,CO,CO-ANT,Antioquía Department,Amalfi,0,AFI
6098,SKAP,SKAP,SKAP,APY,SKAP,icao_code,Gomez Nino Apiay ...,medium_airport,4.07607,-73.5627,1207,CO,CO-MET,Meta Department,Apiay,0,API
6099,SKAR,SKAR,SKAR,AXM,SKAR,icao_code,El Eden Airport,medium_airport,4.45278,-75.7664,3990,CO,CO-QUI,Quindio Department,Armenia,1,AXM
6100,SKAS,SKAS,SKAS,PUU,SKAS,icao_code,Tres De Mayo Airport,medium_airport,0.505228,-76.5008,815,CO,CO-PUT,Putumayo Department,Puerto Asís,1,PUU
429705,SKAT,SKAT,SKAT,ARQ,SKAT,icao_code,El Troncal Airport,small_airport,7.02106,-71.388901,512,CO,CO-ARA,Arauca Department,Arauquita,0,ARQ


In [25]:
print("Distribución de la columna donde se encontró el código SKXX:")
display(
    dim_airports
    .groupBy("icao_code_source_col")
    .count()
    .orderBy(F.desc("count"))
)

Distribución de la columna donde se encontró el código SKXX:


icao_code_source_col,count
icao_code,108
gps_code,38


In [26]:
print("Ejemplo de descartados sin SKXX válido:")
display(airports_descartados_sin_skxx.limit(10))

Ejemplo de descartados sin SKXX válido:


id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,country_name,iso_country,region_name,iso_region,local_region,municipality,scheduled_service,gps_code,icao_code,iata_code,local_code,home_link,wikipedia_link,keywords,score,last_updated,iso_country_clean,icao_code_final,icao_code_source_col
40906,SK-429,small_airport,Leticia Airport,4.463611,-75.033611,2197,SA,Colombia,CO,Tolima Department,CO-TOL,TOL,Alvarado,0,,,,LCT,,,,50,2009-10-20T10:45:...,CO,NULL,NULL
40593,SK-489,small_airport,San Antonio Airport,0.679722,-70.4375,649,SA,Colombia,CO,Vaupés Department,CO-VAU,VAU,Mitu,0,,,,MSA,,,,50,2009-10-20T10:46:...,CO,NULL,NULL
40797,SK-350,small_airport,Tapa Chicamocha A...,4.416667,-73.4975,5658,SA,Colombia,CO,Meta Department,CO-MET,MET,Villavicencio,0,,,,VVL,,,,50,2009-10-20T10:45:...,CO,NULL,NULL
40887,SK-409,small_airport,El Diamante Airport,4.571667,-74.939167,1268,SA,Colombia,CO,Tolima Department,CO-TOL,TOL,Alvarado,0,,,,ALV,,,,50,2009-10-20T10:45:...,CO,NULL,NULL
40894,SK-446,small_airport,Ventaquemada Airport,4.516667,-74.983333,1700,SA,Colombia,CO,Tolima Department,CO-TOL,TOL,Alvarado,0,,,,VTQ,,,,50,2009-10-20T10:46:...,CO,NULL,NULL
40898,SK-404,small_airport,Calicanto Airport,4.488611,-75.01,2024,SA,Colombia,CO,Tolima Department,CO-TOL,TOL,Alvarado,0,,,,CAT,,,,50,2009-10-20T10:45:...,CO,NULL,NULL
40868,SK-405,small_airport,Cerritos Airport,4.841625,-74.775853,756,SA,Colombia,CO,Tolima Department,CO-TOL,TOL,Ambalema,0,,,,CRR,,,,50,2024-10-25T12:30:...,CO,NULL,NULL
40869,SK-414,small_airport,El Santuario Airport,4.905278,-74.785,750,SA,Colombia,CO,Tolima Department,CO-TOL,TOL,Ambalema,0,,,,AMB,,,,50,2009-10-20T10:45:...,CO,NULL,NULL
40870,SK-415,small_airport,El Triunfo Airport,4.868889,-74.8025,761,SA,Colombia,CO,Tolima Department,CO-TOL,TOL,Ambalema,0,,,,ETR,,,,50,2009-10-20T10:45:...,CO,NULL,NULL
40873,SK-433,small_airport,Pajonales Airport,4.759444,-74.833056,767,SA,Colombia,CO,Tolima Department,CO-TOL,TOL,Ambalema,0,,,,PJS,,,,50,2009-10-20T10:45:...,CO,NULL,NULL


In [27]:
print(f"Primeros 10 de {len(ICAO_CODES)} ICAO Codes: {ICAO_CODES[:10]}")
print(f"Primeros 10 códigos alternativos mapeados a ICAO:\n{list(code_to_icao.items())[:10]}")

Primeros 10 de 146 ICAO Codes: ['SKAA', 'SKAC', 'SKAD', 'SKAG', 'SKAL', 'SKAM', 'SKAP', 'SKAR', 'SKAS', 'SKAT']
Primeros 10 códigos alternativos mapeados a ICAO:
[('SKAA', 'SKAA'), ('SK-054', 'SKAA'), ('PAZ', 'SKAA'), ('SKAC', 'SKAC'), ('ACR', 'SKAC'), ('SKAD', 'SKAD'), ('ACD', 'SKAD'), ('SKAG', 'SKAG'), ('AGH', 'SKAG'), ('HAY', 'SKAG')]


### 7. DESCARGA COMPLETA DE DATOS.GOV.CO

In [28]:
metadata_trafico = get_dataset_metadata(
    resource_id=TRAFICO_OD_RESOURCE,
    app_token=DATOS_GOV_APP_TOKEN,
)

save_json_to_file(
    metadata_trafico,
    BRONZE_METADATA_DIR / "metadata_trafico_origen_destino.json",
)

cols_trafico = metadata_columns_to_spark_df(metadata_trafico)

write_spark_parquet_dataset(
    cols_trafico,
    BRONZE_METADATA_DIR / "columns_trafico_origen_destino.parquet",
)

print("Columnas Tráfico Origen-Destino:")
display(cols_trafico)

[OK] JSON guardado: data_proyecto_aeropuertos/bronze/metadata/metadata_trafico_origen_destino.json
[OK] Path anterior eliminado: data_proyecto_aeropuertos/bronze/metadata/columns_trafico_origen_destino.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/metadata/columns_trafico_origen_destino.parquet | filas=16 | columnas=4 | part-files=8
Columnas Tráfico Origen-Destino:


dataTypeName,description,fieldName,name
text,Sigla OACI con la...,sigla_empresa,Sigla Empresa
text,Nombre que identi...,nombre,Nombre
number,año de la informa...,a_o,Año
number,Mes de la informa...,n_mero_de_mes,Número de Mes
text,Corresponde a la ...,origen,Origen
text,Nombre del aeropu...,nombre_origen,Nombre Origen
text,Ciudad del aeropu...,ciudad_origen,Ciudad Origen
text,País del aeropuer...,pais_origen,Pais Origen
text,Corresponde a la ...,destino,Destino
text,Nombre del aeropu...,nombre_destino,Nombre Destino


In [29]:
metadata_operaciones = get_dataset_metadata(
    resource_id=OPERACIONES_RESOURCE,
    app_token=DATOS_GOV_APP_TOKEN,
)

save_json_to_file(
    metadata_operaciones,
    BRONZE_METADATA_DIR / "metadata_operaciones_aereas.json",
)

cols_operaciones = metadata_columns_to_spark_df(metadata_operaciones)

write_spark_parquet_dataset(
    cols_operaciones,
    BRONZE_METADATA_DIR / "columns_operaciones_aereas.parquet",
)

print("Columnas Operaciones Aéreas:")
display(cols_operaciones)

[OK] JSON guardado: data_proyecto_aeropuertos/bronze/metadata/metadata_operaciones_aereas.json
[OK] Path anterior eliminado: data_proyecto_aeropuertos/bronze/metadata/columns_operaciones_aereas.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/metadata/columns_operaciones_aereas.parquet | filas=11 | columnas=4 | part-files=8
Columnas Operaciones Aéreas:


dataTypeName,description,fieldName,name
text,Periodo cronológi...,anio,ANIO
text,Periodo cronológi...,mes,MES
text,Indicador de luga...,aeropuerto_operacion,AEROPUERTO_OPERACION
text,Punto de partida ...,origen,ORIGEN
text,Punto de llegada ...,destino,DESTINO
text,Designador de la ...,empresa,EMPRESA
text,Poseedor del Cert...,explotador,EXPLOTADOR
text,Clasificación por...,tipo_vuelo,TIPO_VUELO
text,Clasificación geo...,trafico,TRAFICO
text,Naturaleza del mo...,tipo_operacion,TIPO_OPERACION


### 8. DESCARGA DE FILAS DE LOS DOS DATASETS

In [30]:
df_trafico_od_bronze = download_socrata_dataset_all_rows_to_json_pages(
    resource_id=TRAFICO_OD_RESOURCE,
    output_dir=BRONZE_TRAFICO_OD_PAGES_DIR,
    base_name="trafico_origen_destino",
    limit=DATOS_GOV_LIMIT,
    app_token=DATOS_GOV_APP_TOKEN,
    overwrite=OVERWRITE_BRONZE,
)

df_trafico_od = apply_metadata_display_names_spark(
    df_trafico_od_bronze,
    metadata_trafico,
)

write_spark_parquet_dataset(
    df_trafico_od,
    SILVER_TRAFICO_OD_DIR / "trafico_origen_destino_full.parquet",
)

print("Tráfico OD:")
print((df_trafico_od.count(), len(df_trafico_od.columns)))
display(df_trafico_od.limit(10))

[INFO] Total filas esperadas para gb6w-ynu4 (trafico_origen_destino): 455,787
[INFO] Descargando trafico_origen_destino | página 1 | offset 0 | limit 50,000
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/trafico_origen_destino/pages/trafico_origen_destino_page_0001.json
[INFO] Descargando trafico_origen_destino | página 2 | offset 50,000 | limit 50,000
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/trafico_origen_destino/pages/trafico_origen_destino_page_0002.json
[INFO] Descargando trafico_origen_destino | página 3 | offset 100,000 | limit 50,000
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/trafico_origen_destino/pages/trafico_origen_destino_page_0003.json
[INFO] Descargando trafico_origen_destino | página 4 | offset 150,000 | limit 50,000
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/trafico_origen_destino/pages/trafico_origen_destino_page_0004.json
[INFO] Descargando trafico_origen_destino | página 5 | offset 200,000 | limit 50,000
[OK] JSON guardado: data

[OK] Descarga finalizada para trafico_origen_destino. Filas leídas por Spark: 455,787
[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/trafico_origen_destino/trafico_origen_destino_full.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/trafico_origen_destino/trafico_origen_destino_full.parquet | filas=455,787 | columnas=16 | part-files=9
Tráfico OD:


(455787, 16)


Año,Carga_Correo (Kg),Ciudad Destino,Ciudad Origen,Destino,Número de Mes,Nombre,Nombre Destino,Nombre Origen,Origen,Pais Destino,Pais Origen,Pasajeros,Sigla Empresa,Tipo Vuelo,Tráfico (N/I)
2021,0,CARTAGENA,WARSAW,CTG,10,KLM,CARTAGENA - RAFAE...,FEDERIC CHOPN,WAW,COLOMBIA,POLONIA,7,KLM,R,I
2021,0,BOGOTA,ZAGREB,BOG,10,KLM,BOGOTA - EL DORAD...,ZAGREB-PLESO,ZAG,COLOMBIA,CROACIA,37,KLM,R,I
2021,0,CARTAGENA,ZAGREB,CTG,10,KLM,CARTAGENA - RAFAE...,ZAGREB-PLESO,ZAG,COLOMBIA,CROACIA,7,KLM,R,I
2021,0,BOGOTA,ZURICH,BOG,10,KLM,BOGOTA - EL DORAD...,ZURICH,ZRH,COLOMBIA,SUIZA,154,KLM,R,I
2021,0,CARTAGENA,ZURICH,CTG,10,KLM,CARTAGENA - RAFAE...,ZURICH,ZRH,COLOMBIA,SUIZA,48,KLM,R,I
2021,0,BOGOTA,ABENDEEN,BOG,11,KLM,BOGOTA - EL DORAD...,CYDE,ABZ,COLOMBIA,INGLATERRA,6,KLM,R,I
2021,0,CARTAGENA,ABENDEEN,CTG,11,KLM,CARTAGENA - RAFAE...,CYDE,ABZ,COLOMBIA,INGLATERRA,1,KLM,R,I
2021,0,BOGOTA,ACCRA,BOG,11,KLM,BOGOTA - EL DORAD...,KOTAKA INT AIRPORT,ACC,COLOMBIA,GHANA,1,KLM,R,I
2021,0,BOGOTA,MALAGA,BOG,11,KLM,BOGOTA - EL DORAD...,MALAGA,AGP,COLOMBIA,ESPANA,11,KLM,R,I
2021,0,CARTAGENA,MALAGA,CTG,11,KLM,CARTAGENA - RAFAE...,MALAGA,AGP,COLOMBIA,ESPANA,1,KLM,R,I


In [31]:
df_operaciones_bronze = download_socrata_dataset_all_rows_to_json_pages(
    resource_id=OPERACIONES_RESOURCE,
    output_dir=BRONZE_OPERACIONES_PAGES_DIR,
    base_name="operaciones_aereas",
    limit=DATOS_GOV_LIMIT,
    app_token=DATOS_GOV_APP_TOKEN,
    overwrite=OVERWRITE_BRONZE,
)

df_operaciones = apply_metadata_display_names_spark(
    df_operaciones_bronze,
    metadata_operaciones,
)

write_spark_parquet_dataset(
    df_operaciones,
    SILVER_OPERACIONES_DIR / "operaciones_aereas_full.parquet",
)

print("Operaciones:")
print((df_operaciones.count(), len(df_operaciones.columns)))
display(df_operaciones.limit(10))

[INFO] Total filas esperadas para jh8x-n6h6 (operaciones_aereas): 550,724
[INFO] Descargando operaciones_aereas | página 1 | offset 0 | limit 50,000
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/operaciones_aereas/pages/operaciones_aereas_page_0001.json
[INFO] Descargando operaciones_aereas | página 2 | offset 50,000 | limit 50,000
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/operaciones_aereas/pages/operaciones_aereas_page_0002.json
[INFO] Descargando operaciones_aereas | página 3 | offset 100,000 | limit 50,000
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/operaciones_aereas/pages/operaciones_aereas_page_0003.json
[INFO] Descargando operaciones_aereas | página 4 | offset 150,000 | limit 50,000
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/operaciones_aereas/pages/operaciones_aereas_page_0004.json
[INFO] Descargando operaciones_aereas | página 5 | offset 200,000 | limit 50,000
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/operaciones_aereas/pages/op

[OK] Descarga finalizada para operaciones_aereas. Filas leídas por Spark: 550,724
[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/operaciones_aereas/operaciones_aereas_full.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/operaciones_aereas/operaciones_aereas_full.parquet | filas=550,724 | columnas=11 | part-files=11
Operaciones:
(550724, 11)


AEROPUERTO_OPERACION,ANIO,DESTINO,EMPRESA,EXPLOTADOR,MES,ORIGEN,TIPO_OPERACION,TIPO_VUELO,TotalOperaciones,TRAFICO
SKAP,2017,SKAP,Estado,Estado,1,SKAP,LLEGADA,M,1,N
SKAP,2017,SKAP,Estado,Estado,1,SKAP,SALIDA,M,1,N
SKAP,2017,SKBO,Estado,Estado,1,SKAP,SALIDA,M,22,N
SKAP,2017,SKBQ,Estado,Estado,1,SKAP,SALIDA,M,1,N
SKAP,2017,SKGB,Estado,Estado,1,SKAP,SALIDA,M,1,N
SKAP,2017,SKMU,Estado,Estado,1,SKAP,SALIDA,M,2,N
SKAP,2017,SKOE,Estado,Estado,1,SKAP,SALIDA,M,1,N
SKAP,2017,SKPQ,Estado,Estado,1,SKAP,SALIDA,M,1,N
SKAP,2017,SKRG,Estado,Estado,1,SKAP,SALIDA,M,1,N
SKAP,2017,SKSA,Estado,Estado,1,SKAP,SALIDA,M,2,N


### 9. FUNCIONES DE TRANSFORMACIÓN PARA DATOS.GOV.CO

In [32]:
def code_to_icao_expr(input_col: str, code_to_icao: dict, allow_direct_skxx: bool = True):
    clean = normalizar_codigo_expr(input_col)

    mapping_expr = (
        F.create_map(
            *[
                item
                for kv in code_to_icao.items()
                for item in (F.lit(kv[0]), F.lit(kv[1]))
            ]
        )
        if code_to_icao
        else F.create_map()
    )

    mapped = F.element_at(mapping_expr, clean)

    if allow_direct_skxx:
        return (
            F.when(mapped.isNotNull(), mapped)
            .when(es_codigo_skxx_valido_expr(clean), clean)
            .otherwise(F.lit(None).cast("string"))
        )

    return (
        F.when(mapped.isNotNull(), mapped)
        .otherwise(F.lit(None).cast("string"))
    )

In [33]:
def construir_operaciones_airport_month_spark(
    df_operaciones: DataFrame,
    code_to_icao: dict,
) -> tuple[DataFrame, DataFrame]:
    df = df_operaciones

    anio_col = find_column_spark(df, ["ANIO", "AÑO", "anio", "año", "ano"], label="año operaciones")
    mes_col = find_column_spark(df, ["MES", "mes"], label="mes operaciones")
    aeropuerto_col = find_column_spark(df, ["AEROPUERTO_OPERACION", "aeropuerto_operacion", "Aeropuerto Operacion", "Aeropuerto Operación"], label="aeropuerto operación")
    total_col = find_column_spark(df, ["TotalOperaciones", "total_operaciones", "Total Operaciones", "operaciones", "cantidad_operaciones"], label="total operaciones")
    tipo_operacion_col = find_column_spark(df, ["TIPO_OPERACION", "tipo_operacion", "Tipo Operacion", "Tipo Operación"], required=False, label="tipo operación")

    df = crear_fecha_mes_desde_anio_mes_spark(df, anio_col, mes_col)

    df = df.filter(
        (F.col("fecha_mes") >= F.to_date(F.lit(MODEL_START_MONTH))) &
        (F.col("fecha_mes") <= F.to_date(F.lit(MODEL_END_MONTH)))
    )

    df = (
        df
        .withColumn("aeropuerto_operacion_original", F.upper(F.trim(F.col(aeropuerto_col).cast("string"))))
        .withColumn("icao_code", code_to_icao_expr(aeropuerto_col, code_to_icao))
    )

    no_mapeados = (
        df
        .filter(F.col("icao_code").isNull())
        .groupBy("aeropuerto_operacion_original")
        .count()
        .withColumnRenamed("count", "n_filas")
        .orderBy(F.desc("n_filas"))
    )

    df = (
        df
        .filter(F.col("icao_code").isNotNull())
        .withColumn("TotalOperaciones", F.coalesce(to_numeric_expr(total_col), F.lit(0.0)))
    )

    if tipo_operacion_col is not None:
        df = df.withColumn(
            "tipo_operacion_clean",
            F.upper(F.trim(F.col(tipo_operacion_col).cast("string"))),
        )

        ops_pivot = (
            df
            .groupBy("icao_code", "fecha_mes")
            .pivot("tipo_operacion_clean")
            .agg(F.sum("TotalOperaciones"))
            .fillna(0)
        )

        rename_map = {}

        for col in ops_pivot.columns:
            col_key = normalize_text_key(col)

            if col_key in ["llegada", "llegadas", "arrival", "arrivals"]:
                rename_map[col] = "operaciones_llegada"

            if col_key in ["salida", "salidas", "departure", "departures"]:
                rename_map[col] = "operaciones_salida"

        for src, dst in rename_map.items():
            ops_pivot = ops_pivot.withColumnRenamed(src, dst)

        if "operaciones_llegada" not in ops_pivot.columns:
            ops_pivot = ops_pivot.withColumn("operaciones_llegada", F.lit(0.0))

        if "operaciones_salida" not in ops_pivot.columns:
            ops_pivot = ops_pivot.withColumn("operaciones_salida", F.lit(0.0))

        numeric_extra_cols = [
            col for col in ops_pivot.columns
            if col not in ["icao_code", "fecha_mes", "operaciones_llegada", "operaciones_salida"]
        ]

        if numeric_extra_cols:
            from functools import reduce
            import operator

            ops_pivot = ops_pivot.withColumn(
                "operaciones_otras",
                reduce(operator.add, [F.col(col).cast("double") for col in numeric_extra_cols]),
            )
        else:
            ops_pivot = ops_pivot.withColumn("operaciones_otras", F.lit(0.0))

        operations_airport_month = ops_pivot.withColumn(
            "operaciones_total",
            F.col("operaciones_llegada") +
            F.col("operaciones_salida") +
            F.col("operaciones_otras"),
        )

    else:
        operations_airport_month = (
            df
            .groupBy("icao_code", "fecha_mes")
            .agg(F.sum("TotalOperaciones").alias("operaciones_total"))
            .withColumn("operaciones_llegada", F.lit(None).cast("double"))
            .withColumn("operaciones_salida", F.lit(None).cast("double"))
            .withColumn("operaciones_otras", F.lit(None).cast("double"))
        )

    operations_airport_month = operations_airport_month.orderBy("icao_code", "fecha_mes")

    return operations_airport_month, no_mapeados

In [34]:
def construir_od_airport_month_spark(
    df_transporte: DataFrame,
    code_to_icao: dict,
) -> tuple[DataFrame, DataFrame, DataFrame]:
    df = df_transporte

    anio_col = find_column_spark(df, ["Año", "ANIO", "AÑO", "anio", "año", "ano"], label="año tráfico OD")
    mes_col = find_column_spark(df, ["Número de Mes", "Numero de Mes", "NÚMERO DE MES", "numero_de_mes", "mes"], label="mes tráfico OD")

    origen_col = find_column_spark(df, ["Origen", "origen"], label="origen")
    destino_col = find_column_spark(df, ["Destino", "destino"], label="destino")

    pasajeros_col = find_column_spark(df, ["Pasajeros", "pasajeros"], label="pasajeros")
    carga_col = find_column_spark(df, ["Carga_Correo (Kg)", "Carga Correo Kg", "carga_correo_kg", "Carga y Correo", "carga"], label="carga/correo")

    empresa_col = find_column_spark(df, ["Sigla Empresa", "sigla_empresa", "Empresa", "empresa"], required=False, label="empresa")
    trafico_col = find_column_spark(df, ["Tráfico (N/I)", "Trafico (N/I)", "trafico_n_i", "Tráfico", "Trafico"], required=False, label="tráfico N/I")

    df = crear_fecha_mes_desde_anio_mes_spark(df, anio_col, mes_col)

    df = df.filter(
        (F.col("fecha_mes") >= F.to_date(F.lit(MODEL_START_MONTH))) &
        (F.col("fecha_mes") <= F.to_date(F.lit(MODEL_END_MONTH)))
    )

    df = (
        df
        .withColumn("origen_original", F.upper(F.trim(F.col(origen_col).cast("string"))))
        .withColumn("destino_original", F.upper(F.trim(F.col(destino_col).cast("string"))))
        .withColumn("icao_origen", code_to_icao_expr(origen_col, code_to_icao))
        .withColumn("icao_destino", code_to_icao_expr(destino_col, code_to_icao))
    )

    origenes_no_mapeados = (
        df
        .filter(F.col("icao_origen").isNull())
        .groupBy("origen_original")
        .count()
        .withColumnRenamed("count", "n_filas")
        .orderBy(F.desc("n_filas"))
    )

    destinos_no_mapeados = (
        df
        .filter(F.col("icao_destino").isNull())
        .groupBy("destino_original")
        .count()
        .withColumnRenamed("count", "n_filas")
        .orderBy(F.desc("n_filas"))
    )

    df = (
        df
        .filter(F.col("icao_origen").isNotNull() | F.col("icao_destino").isNotNull())
        .withColumn("Pasajeros", F.coalesce(to_numeric_expr(pasajeros_col), F.lit(0.0)))
        .withColumn("Carga_Correo_Kg", F.coalesce(to_numeric_expr(carga_col), F.lit(0.0)))
    )

    if trafico_col is not None:
        df = df.withColumn("trafico_clean", F.upper(F.trim(F.col(trafico_col).cast("string"))))
    else:
        df = df.withColumn("trafico_clean", F.lit("NO_DISPONIBLE"))

    if empresa_col is None:
        df = df.withColumn("_empresa_tmp", F.lit("NO_DISPONIBLE"))
        empresa_col = "_empresa_tmp"

    od_salidas = (
        df
        .filter(F.col("icao_origen").isNotNull())
        .groupBy("icao_origen", "fecha_mes")
        .agg(
            F.sum("Pasajeros").alias("pasajeros_salida"),
            F.sum("Carga_Correo_Kg").alias("carga_salida_kg"),
            F.countDistinct(destino_col).alias("n_destinos_total"),
            F.countDistinct(empresa_col).alias("n_empresas_salida"),
            F.count(destino_col).alias("n_registros_salida"),
        )
        .withColumnRenamed("icao_origen", "icao_code")
    )

    od_llegadas = (
        df
        .filter(F.col("icao_destino").isNotNull())
        .groupBy("icao_destino", "fecha_mes")
        .agg(
            F.sum("Pasajeros").alias("pasajeros_llegada"),
            F.sum("Carga_Correo_Kg").alias("carga_llegada_kg"),
            F.countDistinct(origen_col).alias("n_origenes_total"),
            F.countDistinct(empresa_col).alias("n_empresas_llegada"),
            F.count(origen_col).alias("n_registros_llegada"),
        )
        .withColumnRenamed("icao_destino", "icao_code")
    )

    od_airport_month = od_salidas.join(
        od_llegadas,
        on=["icao_code", "fecha_mes"],
        how="outer",
    )

    numeric_cols = [
        "pasajeros_salida",
        "carga_salida_kg",
        "n_destinos_total",
        "n_empresas_salida",
        "n_registros_salida",
        "pasajeros_llegada",
        "carga_llegada_kg",
        "n_origenes_total",
        "n_empresas_llegada",
        "n_registros_llegada",
    ]

    od_airport_month = od_airport_month.fillna(0, subset=[c for c in numeric_cols if c in od_airport_month.columns])

    od_airport_month = (
        od_airport_month
        .withColumn("pasajeros_total", F.col("pasajeros_salida") + F.col("pasajeros_llegada"))
        .withColumn("carga_total_kg", F.col("carga_salida_kg") + F.col("carga_llegada_kg"))
    )

    trafico_salidas = (
        df
        .filter(F.col("icao_origen").isNotNull())
        .groupBy("icao_origen", "fecha_mes")
        .pivot("trafico_clean")
        .agg(F.sum("Pasajeros"))
        .fillna(0)
        .withColumnRenamed("icao_origen", "icao_code")
    )

    trafico_llegadas = (
        df
        .filter(F.col("icao_destino").isNotNull())
        .groupBy("icao_destino", "fecha_mes")
        .pivot("trafico_clean")
        .agg(F.sum("Pasajeros"))
        .fillna(0)
        .withColumnRenamed("icao_destino", "icao_code")
    )

    for col in trafico_salidas.columns:
        key = normalize_text_key(col)

        if key == "n":
            trafico_salidas = trafico_salidas.withColumnRenamed(col, "pasajeros_salida_nacional")
        elif key == "i":
            trafico_salidas = trafico_salidas.withColumnRenamed(col, "pasajeros_salida_internacional")

    for col in trafico_llegadas.columns:
        key = normalize_text_key(col)

        if key == "n":
            trafico_llegadas = trafico_llegadas.withColumnRenamed(col, "pasajeros_llegada_nacional")
        elif key == "i":
            trafico_llegadas = trafico_llegadas.withColumnRenamed(col, "pasajeros_llegada_internacional")

    trafico_salidas_keep = [
        col for col in trafico_salidas.columns
        if col in ["icao_code", "fecha_mes"] or col.startswith("pasajeros_salida_")
    ]

    trafico_llegadas_keep = [
        col for col in trafico_llegadas.columns
        if col in ["icao_code", "fecha_mes"] or col.startswith("pasajeros_llegada_")
    ]

    od_airport_month = (
        od_airport_month
        .join(trafico_salidas.select(*trafico_salidas_keep), on=["icao_code", "fecha_mes"], how="left")
        .join(trafico_llegadas.select(*trafico_llegadas_keep), on=["icao_code", "fecha_mes"], how="left")
    )

    fill_cols = [
        col for col in od_airport_month.columns
        if col.startswith("pasajeros_") or col.startswith("carga_") or col.startswith("n_")
    ]

    od_airport_month = od_airport_month.fillna(0, subset=fill_cols)

    for missing_col in [
        "pasajeros_salida_internacional",
        "pasajeros_llegada_internacional",
    ]:
        if missing_col not in od_airport_month.columns:
            od_airport_month = od_airport_month.withColumn(missing_col, F.lit(0.0))

    od_airport_month = (
        od_airport_month
        .withColumn(
            "pasajeros_internacional_total",
            F.col("pasajeros_salida_internacional") + F.col("pasajeros_llegada_internacional"),
        )
        .withColumn(
            "proporcion_pasajeros_internacional",
            safe_divide_expr("pasajeros_internacional_total", "pasajeros_total"),
        )
        .orderBy("icao_code", "fecha_mes")
    )

    return od_airport_month, origenes_no_mapeados, destinos_no_mapeados

### 10. CONSTRUIR OPERACIONES Y ORIGEN-DESTINO A NIVEL AEROPUERTO-MES

In [35]:
operations_airport_month, operaciones_no_mapeadas = construir_operaciones_airport_month_spark(
    df_operaciones=df_operaciones,
    code_to_icao=code_to_icao,
)

write_spark_parquet_dataset(
    operations_airport_month,
    SILVER_OPERACIONES_DIR / "operations_airport_month.parquet",
)

write_spark_parquet_dataset(
    operaciones_no_mapeadas,
    SILVER_DIAGNOSTICS_DIR / "operaciones_aeropuertos_no_mapeados.parquet",
)

print("Operaciones aeropuerto-mes:")
display(operations_airport_month.limit(10))

print("Operaciones no mapeadas:")
display(operaciones_no_mapeadas.limit(20))

[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/operaciones_aereas/operations_airport_month.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/operaciones_aereas/operations_airport_month.parquet | filas=8,673 | columnas=7 | part-files=1
[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/diagnostics/operaciones_aeropuertos_no_mapeados.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/operaciones_aeropuertos_no_mapeados.parquet | filas=91 | columnas=2 | part-files=1
Operaciones aeropuerto-mes:


icao_code,fecha_mes,operaciones_llegada,operaciones_salida,TOQUE Y DESPEGUE,operaciones_otras,operaciones_total
SKAC,2020-01-01,5.0,2.0,0.0,0.0,7.0
SKAC,2020-02-01,3.0,0.0,0.0,0.0,3.0
SKAC,2020-04-01,2.0,1.0,0.0,0.0,3.0
SKAC,2020-05-01,1.0,0.0,0.0,0.0,1.0
SKAC,2020-06-01,4.0,2.0,0.0,0.0,6.0
SKAC,2020-07-01,1.0,1.0,0.0,0.0,2.0
SKAC,2020-08-01,1.0,0.0,0.0,0.0,1.0
SKAC,2020-09-01,5.0,0.0,0.0,0.0,5.0
SKAC,2020-10-01,0.0,1.0,0.0,0.0,1.0
SKAC,2020-11-01,3.0,0.0,0.0,0.0,3.0


Operaciones no mapeadas:


aeropuerto_operacion_original,n_filas
SQZJ,362
SQGJ,335
SQZN,98
SQZP,45
SQLY,37
SQOT,36
SQQK,36
SQRN,26
SQPK,24
SQKS,21


In [36]:
od_airport_month, origenes_no_mapeados, destinos_no_mapeados = construir_od_airport_month_spark(
    df_transporte=df_trafico_od,
    code_to_icao=code_to_icao,
)

write_spark_parquet_dataset(
    od_airport_month,
    SILVER_TRAFICO_OD_DIR / "od_airport_month.parquet",
)

write_spark_parquet_dataset(
    origenes_no_mapeados,
    SILVER_DIAGNOSTICS_DIR / "trafico_origenes_no_mapeados.parquet",
)

write_spark_parquet_dataset(
    destinos_no_mapeados,
    SILVER_DIAGNOSTICS_DIR / "trafico_destinos_no_mapeados.parquet",
)

print("OD aeropuerto-mes:")
display(od_airport_month.limit(10))

print("Orígenes no mapeados:")
display(origenes_no_mapeados.limit(20))

print("Destinos no mapeados:")
display(destinos_no_mapeados.limit(20))

[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/trafico_origen_destino/od_airport_month.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/trafico_origen_destino/od_airport_month.parquet | filas=5,898 | columnas=20 | part-files=1
[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/diagnostics/trafico_origenes_no_mapeados.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/trafico_origenes_no_mapeados.parquet | filas=1,030 | columnas=2 | part-files=1
[OK] Path anterior eliminado: data_proyecto_aeropuertos/silver/diagnostics/trafico_destinos_no_mapeados.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/trafico_destinos_no_mapeados.parquet | filas=1,025 | columnas=2 | part-files=1
OD aeropuerto-mes:


icao_code,fecha_mes,pasajeros_salida,carga_salida_kg,n_destinos_total,n_empresas_salida,n_registros_salida,pasajeros_llegada,carga_llegada_kg,n_origenes_total,n_empresas_llegada,n_registros_llegada,pasajeros_total,carga_total_kg,pasajeros_salida_internacional,pasajeros_salida_nacional,pasajeros_llegada_internacional,pasajeros_llegada_nacional,pasajeros_internacional_total,proporcion_pasajeros_internacional
SKAA,2020-03-01,1.0,0.0,1,1,1,0.0,0.0,0,0,0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
SKAA,2021-08-01,9.0,0.0,1,1,1,0.0,0.0,0,0,0,9.0,0.0,9.0,0.0,0.0,0.0,9.0,1.0
SKAA,2022-11-01,0.0,10.0,1,1,1,3.0,40.0,1,1,1,3.0,50.0,0.0,0.0,3.0,0.0,3.0,1.0
SKAA,2023-03-01,1.0,0.0,1,1,1,1.0,0.0,1,1,1,2.0,0.0,1.0,0.0,1.0,0.0,2.0,1.0
SKAC,2020-01-01,138.0,17740.0,5,5,8,115.0,9310.0,5,5,7,253.0,27050.0,0.0,138.0,0.0,115.0,0.0,0.0
SKAC,2020-02-01,60.0,27868.0,3,7,9,70.0,20505.0,4,7,11,130.0,48373.0,0.0,60.0,0.0,70.0,0.0,0.0
SKAC,2020-03-01,105.0,18554.0,3,4,5,111.0,9054.0,3,4,5,216.0,27608.0,0.0,105.0,0.0,111.0,0.0,0.0
SKAC,2020-04-01,4.0,12988.0,2,3,3,0.0,3455.0,4,3,4,4.0,16443.0,0.0,4.0,0.0,0.0,0.0,0.0
SKAC,2020-05-01,1.0,21328.0,3,4,4,1.0,13278.0,5,4,8,2.0,34606.0,0.0,1.0,0.0,1.0,0.0,0.0
SKAC,2020-06-01,0.0,12738.0,1,3,3,2.0,16910.0,4,3,5,2.0,29648.0,0.0,0.0,0.0,2.0,0.0,0.0


Orígenes no mapeados:


origen_original,n_filas
MIA,4795
PTY,2579
MEX,2412
LIM,2320
UIO,2095
JFK,1997
LAX,1992
CUN,1962
SCL,1941
MCO,1923


Destinos no mapeados:


destino_original,n_filas
MIA,4823
PTY,3133
MEX,3028
LIM,2872
UIO,2726
JFK,2637
CUN,2545
SCL,2534
MAD,2527
MCO,2426


---

### **11. DEFINIR AEROPUERTOS PARA DESCARGA IEM**

In [38]:
icaos_con_operaciones = {
    row["icao_code"]
    for row in operations_airport_month.select("icao_code").distinct().collect()
    if row["icao_code"] is not None
}

icaos_con_od = {
    row["icao_code"]
    for row in od_airport_month.select("icao_code").distinct().collect()
    if row["icao_code"] is not None
}

icaos_para_iem = sorted(icaos_con_operaciones & icaos_colombia_set)

if MAX_IEM_AIRPORTS is not None:
    icaos_para_iem = icaos_para_iem[:MAX_IEM_AIRPORTS]

diagnostico_icaos = {
    "n_icaos_colombia_validos": len(icaos_colombia_set),
    "n_icaos_con_operaciones": len(icaos_con_operaciones),
    "n_icaos_con_od": len(icaos_con_od),
    "n_icaos_para_iem": len(icaos_para_iem),
    "icaos_para_iem": icaos_para_iem,
}

save_json_to_file(
    diagnostico_icaos,
    SILVER_DIAGNOSTICS_DIR / "diagnostico_icaos_para_iem.json",
)

print(json.dumps(diagnostico_icaos, indent=2, ensure_ascii=False))

[OK] JSON guardado: data_proyecto_aeropuertos/silver/diagnostics/diagnostico_icaos_para_iem.json
{
  "n_icaos_colombia_validos": 146,
  "n_icaos_con_operaciones": 245,
  "n_icaos_con_od": 113,
  "n_icaos_para_iem": 139,
  "icaos_para_iem": [
    "SKAC",
    "SKAD",
    "SKAG",
    "SKAL",
    "SKAM",
    "SKAP",
    "SKAR",
    "SKAS",
    "SKAT",
    "SKBC",
    "SKBE",
    "SKBG",
    "SKBM",
    "SKBN",
    "SKBO",
    "SKBQ",
    "SKBR",
    "SKBS",
    "SKBU",
    "SKCA",
    "SKCB",
    "SKCC",
    "SKCD",
    "SKCE",
    "SKCG",
    "SKCI",
    "SKCL",
    "SKCM",
    "SKCN",
    "SKCO",
    "SKCP",
    "SKCR",
    "SKCU",
    "SKCV",
    "SKCZ",
    "SKEB",
    "SKEH",
    "SKEJ",
    "SKFE",
    "SKFL",
    "SKFR",
    "SKFU",
    "SKGA",
    "SKGB",
    "SKGI",
    "SKGO",
    "SKGP",
    "SKGY",
    "SKGZ",
    "SKHA",
    "SKHC",
    "SKHZ",
    "SKIB",
    "SKIG",
    "SKIM",
    "SKIO",
    "SKIP",
    "SKIR",
    "SKJC",
    "SKJU",
    "SKLA",
    "SKLC",
    "SKLG",
  

### 12. DESCARGA IEM HISTÓRICO 2020-01 A 2025-12

In [39]:
def build_iem_params(station: str, start_z: str, end_z: str) -> dict:
    station = station.strip().upper()

    if not station:
        raise ValueError("La estación IEM no puede estar vacía.")

    return {
        "station": station,
        "data": [
            "metar",
            "tmpf",
            "dwpf",
            "relh",
            "drct",
            "sknt",
            "vsby",
            "gust",
            "skyc1",
            "skyl1",
            "wxcodes",
            "alti",
        ],
        "sts": start_z,
        "ets": end_z,
        "tz": "UTC",
        "format": "onlycomma",
        "latlon": "yes",
        "elev": "yes",
        "missing": "null",
        "trace": "null",
        "direct": "yes",
    }

In [40]:
def download_iem_metar_for_airport(
    icao: str,
    iem_start_z: str = IEM_START,
    iem_end_exclusive_z: str = IEM_END_EXCLUSIVE,
    overwrite: bool = True,
) -> None:
    icao = str(icao).strip().upper()

    paths = get_airport_paths(icao)

    output_path = (
        paths["bronze_iem_metar_dir"] /
        "historical_2020-01-01_2025-12-31.parquet"
    )

    metadata_path = (
        paths["bronze_iem_metar_dir"] /
        "historical_2020-01-01_2025-12-31_metadata.json"
    )

    params = build_iem_params(
        station=icao,
        start_z=iem_start_z,
        end_z=iem_end_exclusive_z,
    )

    df_iem_bronze = download_csv_as_spark_parquet(
        service_name="iem",
        url=IEM_ASOS_URL,
        output_path=output_path,
        params=params,
        overwrite=overwrite,
        spark_read_options={
            "inferSchema": "false",
            "nullValue": "null",
        },
    )

    save_json_to_file(
        {
            "icao_code": icao,
            "source": "iem_metar",
            "storage_format": "parquet",
            "original_response_format": "csv",
            "window_start_utc": iem_start_z,
            "window_end_exclusive_utc": iem_end_exclusive_z,
            "model_month_start": MODEL_START_MONTH,
            "model_month_end": MODEL_END_MONTH,
            "n_rows_downloaded": int(df_iem_bronze.count()),
            "n_columns_downloaded": int(len(df_iem_bronze.columns)),
            "output_path": str(output_path),
        },
        metadata_path,
    )

In [41]:
if RUN_IEM_DOWNLOAD:
    for idx, icao in enumerate(icaos_para_iem, start=1):
        print(f"[IEM] {idx}/{len(icaos_para_iem)} descargando {icao}...")
        download_iem_metar_for_airport(
            icao=icao,
            iem_start_z=IEM_START,
            iem_end_exclusive_z=IEM_END_EXCLUSIVE,
            overwrite=OVERWRITE_BRONZE,
        )

    print("[OK] Descarga IEM completada.")
else:
    print("[INFO] RUN_IEM_DOWNLOAD=False. Se omite descarga y se usarán archivos existentes.")

[IEM] 1/139 descargando SKAC...
[OK] Path anterior eliminado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAC/metar/historical_2020-01-01_2025-12-31.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAC/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17 | part-files=1
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAC/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAC/metar/historical_2020-01-01_2025-12-31_metadata.json
[IEM] 2/139 descargando SKAD...
[OK] Path anterior eliminado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAD/metar/historical_2020-01-01_2025-12-31.parquet
[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAD/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17 | part-files=1
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAD/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17
[OK] JSON gu

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAP/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17 | part-files=1
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAP/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAP/metar/historical_2020-01-01_2025-12-31_metadata.json
[IEM] 7/139 descargando SKAR...
[OK] Path anterior eliminado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAR/metar/historical_2020-01-01_2025-12-31.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAR/metar/historical_2020-01-01_2025-12-31.parquet | filas=33,913 | columnas=17 | part-files=8
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAR/metar/historical_2020-01-01_2025-12-31.parquet | filas=33,913 | columnas=17
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAR/metar/historical_2020-01-01_2025-12-31_metadata.json
[IEM] 8/139 descargando SKAS...
[OK] Path anterior eliminado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAS/metar/historical_2020-01-01_2025-12-31.parquet


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAS/metar/historical_2020-01-01_2025-12-31.parquet | filas=10,081 | columnas=17 | part-files=8
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAS/metar/historical_2020-01-01_2025-12-31.parquet | filas=10,081 | columnas=17
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAS/metar/historical_2020-01-01_2025-12-31_metadata.json
[IEM] 9/139 descargando SKAT...
[OK] Path anterior eliminado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKAT/metar/historical_2020-01-01_2025-12-31.parquet


...

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVL/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17 | part-files=1
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVL/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVL/metar/historical_2020-01-01_2025-12-31_metadata.json
[IEM] 136/139 descargando SKVN...


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVN/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17 | part-files=1
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVN/metar/historical_2020-01-01_2025-12-31.parquet | filas=0 | columnas=17
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVN/metar/historical_2020-01-01_2025-12-31_metadata.json
[IEM] 137/139 descargando SKVP...


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVP/metar/historical_2020-01-01_2025-12-31.parquet | filas=31,808 | columnas=17 | part-files=8
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVP/metar/historical_2020-01-01_2025-12-31.parquet | filas=31,808 | columnas=17
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVP/metar/historical_2020-01-01_2025-12-31_metadata.json
[IEM] 138/139 descargando SKVV...


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVV/metar/historical_2020-01-01_2025-12-31.parquet | filas=28,463 | columnas=17 | part-files=8
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVV/metar/historical_2020-01-01_2025-12-31.parquet | filas=28,463 | columnas=17
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKVV/metar/historical_2020-01-01_2025-12-31_metadata.json
[IEM] 139/139 descargando SKYP...


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKYP/metar/historical_2020-01-01_2025-12-31.parquet | filas=10,069 | columnas=17 | part-files=8
[OK] CSV descargado y convertido a Parquet Spark: data_proyecto_aeropuertos/bronze/iem/by_airport/SKYP/metar/historical_2020-01-01_2025-12-31.parquet | filas=10,069 | columnas=17
[OK] JSON guardado: data_proyecto_aeropuertos/bronze/iem/by_airport/SKYP/metar/historical_2020-01-01_2025-12-31_metadata.json
[OK] Descarga IEM completada.


#### 12.1. VERIFICAR DATASETS `.parquet` DE IEM COMO UNIDADES LÓGICAS

In [43]:
def get_path_size_bytes(path: Path) -> int:
    """
    Calcula el tamaño físico total de un path.

    Si el path es un archivo, retorna su tamaño.
    Si el path es una carpeta Spark .parquet, suma el tamaño de todos sus archivos internos.
    """
    path = Path(path)

    if not path.exists():
        return 0

    if path.is_file():
        return int(path.stat().st_size)

    total_size = 0

    for file_path in path.rglob("*"):
        if file_path.is_file():
            total_size += int(file_path.stat().st_size)

    return total_size


def listar_parquets_logicos(base_dir: Path) -> list[Path]:
    """
    Lista Parquets como datasets lógicos.

    Soporta dos casos:
    1. Pandas/pyarrow:
        archivo.parquet

    2. Spark:
        archivo.parquet/
        ├── part-00000-...
        ├── part-00001-...
        └── _SUCCESS

    La función NO cuenta los part-*.parquet internos como datasets independientes.
    """
    base_dir = Path(base_dir)

    if not base_dir.exists():
        return []

    resultados = []

    def recorrer(path: Path) -> None:
        if path.name.endswith(".parquet"):
            resultados.append(path)
            return

        if path.is_dir():
            for child in sorted(path.iterdir()):
                recorrer(child)

    recorrer(base_dir)

    return resultados


def contar_filas_parquet(path_parquet: Path) -> int:
    """
    Cuenta filas de un dataset Parquet lógico usando Spark.

    Funciona tanto para:
    - archivo único .parquet
    - carpeta Spark .parquet con part-files
    """
    path_parquet = Path(path_parquet)

    try:
        return read_spark_parquet(path_parquet).count()
    except Exception as exc:
        print(f"[WARN] No se pudo contar {path_parquet}: {exc}")
        return -1


def diagnosticar_iem_parquet_spark(base_dir: Path) -> DataFrame:
    """
    Diagnostica datasets Parquet de IEM como unidades lógicas.

    No cuenta cada part-file Spark como un archivo independiente.
    """
    schema = T.StructType([
        T.StructField("path", T.StringType(), True),
        T.StructField("n_filas", T.LongType(), True),
        T.StructField("size_bytes", T.LongType(), True),
        T.StructField("estado", T.StringType(), True),
        T.StructField("tipo_parquet", T.StringType(), True),
    ])

    base_dir = Path(base_dir)

    if not base_dir.exists():
        print(f"No existe: {base_dir}")
        return spark.createDataFrame([], schema=schema)

    registros = []

    for parquet_path in sorted(listar_parquets_logicos(base_dir)):
        size_bytes = get_path_size_bytes(parquet_path)

        n_filas = contar_filas_parquet(parquet_path)

        if n_filas < 0:
            estado = "error_lectura"
        elif n_filas == 0:
            estado = "parquet_sin_filas"
        else:
            estado = "parquet_con_datos"

        tipo_parquet = (
            "spark_dataset_dir"
            if parquet_path.is_dir()
            else "single_file_parquet"
        )

        registros.append({
            "path": str(parquet_path),
            "n_filas": int(max(n_filas, 0)),
            "size_bytes": int(size_bytes),
            "estado": estado,
            "tipo_parquet": tipo_parquet,
        })

    return spark.createDataFrame(registros, schema=schema)


In [44]:


df_diagnostico_iem_parquet = diagnosticar_iem_parquet_spark(
    BRONZE_IEM_BY_AIRPORT_DIR
)

print(f"Total datasets Parquet lógicos procesados: {df_diagnostico_iem_parquet.count()}")

print("Resumen IEM Parquet por estado:")
display(
    df_diagnostico_iem_parquet
    .groupBy("estado", "tipo_parquet")
    .agg(
        F.count("path").alias("n_datasets_parquet"),
        F.sum("n_filas").alias("filas_totales"),
        F.sum("size_bytes").alias("size_total_bytes"),
    )
    .orderBy(F.desc("n_datasets_parquet"))
)

path_lista_con_filas = [
    row["path"]
    for row in df_diagnostico_iem_parquet
    .filter(F.col("estado") == "parquet_con_datos")
    .select("path")
    .collect()
]

path_lista_sin_filas = [
    row["path"]
    for row in df_diagnostico_iem_parquet
    .filter(F.col("estado") == "parquet_sin_filas")
    .select("path")
    .collect()
]

print(f"Con datos: {len(path_lista_con_filas)}")
print(f"Sin datos: {len(path_lista_sin_filas)}")

Total datasets Parquet lógicos procesados: 139
Resumen IEM Parquet por estado:


estado,tipo_parquet,n_datasets_parquet,filas_totales,size_total_bytes
parquet_sin_filas,spark_dataset_dir,91,0,152971
parquet_con_datos,spark_dataset_dir,48,1215169,38788479


Con datos: 48
Sin datos: 91


#### 12.2. DIAGNÓSTICO GENERAL DE ARCHIVOS Y DATASETS DEL PROYECTO

In [46]:
def diagnosticar_archivos_proyecto_spark(
    base_dir: Path,
    excluir_dirs: list[str] | None = None,
) -> DataFrame:
    """
    Diagnostica archivos del proyecto evitando carpetas auxiliares.

    Se excluyen carpetas de backup, temporales y resultados para evitar
    errores al intentar leer Parquet generados por pandas/pyarrow con tipos
    no compatibles con Spark, como TIMESTAMP(NANOS).
    """

    schema = T.StructType([
        T.StructField("path", T.StringType(), True),
        T.StructField("nombre_archivo", T.StringType(), True),
        T.StructField("extension", T.StringType(), True),
        T.StructField("size_bytes", T.LongType(), True),
        T.StructField("estado", T.StringType(), True),
        T.StructField("n_filas", T.LongType(), True),
        T.StructField("n_columnas", T.LongType(), True),
        T.StructField("tipo_elemento", T.StringType(), True),
    ])

    base_dir = Path(base_dir)

    if excluir_dirs is None:
        excluir_dirs = [
            "_backup",
            "backup",
            "backup_pandas",
            "_tmp",
            "tmp",
            "results",
            "model_artifacts",
            "__pycache__",
        ]

    if not base_dir.exists():
        return spark.createDataFrame([], schema=schema)

    registros = []

    def debe_excluir_path(path: Path) -> bool:
        """
        Determina si una ruta debe excluirse del diagnóstico.
        """

        partes = [p.lower() for p in path.parts]

        for parte in partes:
            for patron in excluir_dirs:
                if parte.startswith(patron.lower()):
                    return True

        return False

    def diagnosticar_parquet(path: Path) -> dict:
        """
        Diagnostica un archivo o carpeta Parquet de forma defensiva.
        """

        size_bytes = get_path_size_bytes(path)

        tipo_elemento = (
            "spark_dataset_dir"
            if path.is_dir()
            else "single_file_parquet"
        )

        try:
            df_tmp = read_spark_parquet(path)

            n_columnas = len(df_tmp.columns)
            n_filas = df_tmp.count()

            estado = (
                "parquet_sin_filas"
                if n_filas == 0
                else "parquet_con_datos"
            )

        except Exception as exc:
            estado = f"parquet_error_lectura_{type(exc).__name__}"
            n_filas = 0
            n_columnas = None

            print(f"[WARN] No se pudo leer Parquet: {path}")
            print(f"       Error: {type(exc).__name__}: {str(exc)[:300]}")

        return {
            "path": str(path),
            "nombre_archivo": path.name,
            "extension": ".parquet",
            "size_bytes": int(size_bytes),
            "estado": estado,
            "n_filas": int(n_filas),
            "n_columnas": int(n_columnas) if n_columnas is not None else None,
            "tipo_elemento": tipo_elemento,
        }

    def recorrer(path: Path) -> None:
        """
        Recorre recursivamente el proyecto.
        """

        if debe_excluir_path(path):
            return

        if path.name.endswith(".parquet"):
            registros.append(
                diagnosticar_parquet(path)
            )

            return

        if path.is_dir():
            for child in sorted(path.iterdir()):
                recorrer(child)

            return

        if not path.is_file():
            return

        extension = path.suffix.lower()
        size_bytes = int(path.stat().st_size)

        estado = "extension_no_diagnosticada"
        n_filas = None
        n_columnas = None
        tipo_elemento = "file"

        if size_bytes == 0:
            estado = "archivo_fisicamente_vacio"

        elif extension == ".json":
            try:
                with open(path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                if isinstance(data, list):
                    n_filas = len(data)
                    estado = (
                        "json_con_datos"
                        if n_filas > 0
                        else "json_sin_registros"
                    )

                elif isinstance(data, dict):
                    n_filas = 1
                    n_columnas = len(data)
                    estado = "json_con_datos"

                else:
                    estado = f"json_tipo_no_esperado_{type(data).__name__}"

            except Exception as exc:
                estado = f"json_error_lectura_{type(exc).__name__}"

        elif extension == ".csv":
            estado = "csv_no_permitido_en_proyecto"

        registros.append({
            "path": str(path),
            "nombre_archivo": path.name,
            "extension": extension,
            "size_bytes": size_bytes,
            "estado": estado,
            "n_filas": int(n_filas) if n_filas is not None else None,
            "n_columnas": int(n_columnas) if n_columnas is not None else None,
            "tipo_elemento": tipo_elemento,
        })

    recorrer(base_dir)

    return spark.createDataFrame(registros, schema=schema)

In [47]:
df_diagnostico_archivos = diagnosticar_archivos_proyecto_spark(
    BASE_DIR
)

write_spark_parquet_dataset(
    df_diagnostico_archivos,
    SILVER_DIAGNOSTICS_DIR / "diagnostico_archivos_proyecto.parquet",
)

print("Resumen general por estado:")
display(
    df_diagnostico_archivos
    .groupBy("estado", "tipo_elemento")
    .agg(
        F.count("path").alias("n_elementos"),
        F.sum("size_bytes").alias("size_total_bytes"),
        F.sum("n_filas").alias("filas_totales"),
    )
    .orderBy(F.desc("n_elementos"))
)

print("Detalle de datasets Parquet:")
display(
    df_diagnostico_archivos
    .filter(F.col("extension") == ".parquet")
    .select(
        "path",
        "tipo_elemento",
        "estado",
        "n_filas",
        "n_columnas",
        "size_bytes",
    )
    .orderBy("path")
)

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/diagnostico_archivos_proyecto.parquet | filas=317 | columnas=8 | part-files=8
Resumen general por estado:


estado,tipo_elemento,n_elementos,size_total_bytes,filas_totales
json_con_datos,file,165,373482995,1006654
parquet_sin_filas,spark_dataset_dir,91,152971,0
parquet_con_datos,spark_dataset_dir,61,46111040,2239897


Detalle de datasets Parquet:


path,tipo_elemento,estado,n_filas,n_columnas,size_bytes
data_proyecto_aer...,spark_dataset_dir,parquet_con_datos,736,24,121511
data_proyecto_aer...,spark_dataset_dir,parquet_sin_filas,0,17,1681
data_proyecto_aer...,spark_dataset_dir,parquet_sin_filas,0,17,1681
data_proyecto_aer...,spark_dataset_dir,parquet_sin_filas,0,17,1681
data_proyecto_aer...,spark_dataset_dir,parquet_sin_filas,0,17,1681
data_proyecto_aer...,spark_dataset_dir,parquet_sin_filas,0,17,1681
data_proyecto_aer...,spark_dataset_dir,parquet_sin_filas,0,17,1681
data_proyecto_aer...,spark_dataset_dir,parquet_con_datos,33913,17,1123520
data_proyecto_aer...,spark_dataset_dir,parquet_con_datos,10081,17,336287
data_proyecto_aer...,spark_dataset_dir,parquet_sin_filas,0,17,1681


### 13. NORMALIZAR IEM Y AGREGAR A NIVEL MENSUAL

In [48]:
MIN_IEM_ROWS = 500

rows_iem_bronze = []

path_lista_iem_bronze = sorted(
    BRONZE_IEM_BY_AIRPORT_DIR.glob("*/metar/historical_*.parquet")
)

for path_parquet in path_lista_iem_bronze:
    icao_code = path_parquet.parent.parent.name.strip().upper()
    n_rows = contar_filas_parquet(path_parquet)

    rows_iem_bronze.append({
        "icao_code": icao_code,
        "path_parquet": str(path_parquet),
        "n_rows_bronze": int(n_rows),
        "cumple_minimo_500": bool(n_rows >= MIN_IEM_ROWS),
    })

df_iem_bronze_diagnostico = spark.createDataFrame(rows_iem_bronze)

icaos_con_filas = {
    row["icao_code"]
    for row in df_iem_bronze_diagnostico
    .filter(F.col("cumple_minimo_500"))
    .select("icao_code")
    .collect()
}

icaos_para_iem = sorted(
    icaos_con_filas.intersection(icaos_colombia_set)
)

write_spark_parquet_dataset(
    df_iem_bronze_diagnostico,
    SILVER_DIAGNOSTICS_DIR / "iem_bronze_files_diagnostico_min500.parquet",
)

print("Archivos IEM bronze encontrados:", df_iem_bronze_diagnostico.count())
print("ICAO con al menos 500 filas en bronze:", len(icaos_con_filas))
print("ICAO colombianos para pasar a silver:", len(icaos_para_iem))

display(
    df_iem_bronze_diagnostico
    .orderBy(F.desc("n_rows_bronze"))
    .limit(50)
)

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/iem_bronze_files_diagnostico_min500.parquet | filas=139 | columnas=4 | part-files=8
Archivos IEM bronze encontrados: 139
ICAO con al menos 500 filas en bronze: 46
ICAO colombianos para pasar a silver: 46


cumple_minimo_500,icao_code,n_rows_bronze,path_parquet
true,SKBO,58225,data_proyecto_aer...
true,SKCL,56440,data_proyecto_aer...
true,SKRG,55757,data_proyecto_aer...
true,SKBQ,55174,data_proyecto_aer...
true,SKSP,53476,data_proyecto_aer...
true,SKLT,53123,data_proyecto_aer...
true,SKCG,52225,data_proyecto_aer...
true,SKPE,45444,data_proyecto_aer...
true,SKCC,44037,data_proyecto_aer...
true,SKBG,42982,data_proyecto_aer...


In [49]:
def fahrenheit_to_celsius_expr(col_name: str):
    return (to_numeric_expr(col_name) - F.lit(32.0)) * F.lit(5.0 / 9.0)


def normalize_iem_metar_spark(df_iem: DataFrame, icaos_colombia_set: set[str]) -> DataFrame:
    if "station" not in df_iem.columns or "valid" not in df_iem.columns:
        return spark.createDataFrame([], schema=T.StructType([]))

    df = df_iem

    for col in df.columns:
        df = df.withColumn(
            col,
            F.when(F.col(col).cast("string").isin("M", "", "nan", "NaN", "None", "NULL", "null"), F.lit(None))
            .otherwise(F.col(col))
        )

    df = (
        df
        .withColumn("icao_code", F.upper(F.trim(F.col("station").cast("string"))))
        .filter(F.col("icao_code").isin(list(icaos_colombia_set)))
        .withColumn("obs_time_utc", F.to_timestamp(F.col("valid")))
        .filter(F.col("icao_code").isNotNull() & F.col("obs_time_utc").isNotNull())
        .withColumn("fecha_mes", F.to_date(F.date_trunc("month", F.col("obs_time_utc"))))
        .filter(
            (F.col("fecha_mes") >= F.to_date(F.lit(MODEL_START_MONTH))) &
            (F.col("fecha_mes") <= F.to_date(F.lit(MODEL_END_MONTH)))
        )
    )

    if df.limit(1).count() == 0:
        return spark.createDataFrame([], schema=T.StructType([]))

    def optional_numeric(source_col: str, target_col: str):
        nonlocal df
        df = df.withColumn(target_col, to_numeric_expr(source_col) if source_col in df.columns else F.lit(None).cast("double"))

    optional_numeric("lat", "lat")
    optional_numeric("lon", "lon")
    optional_numeric("elevation", "elevation_m")

    df = df.withColumn("bronze_metar", F.col("metar") if "metar" in df.columns else F.lit(None).cast("string"))

    df = df.withColumn("temp_c", fahrenheit_to_celsius_expr("tmpf") if "tmpf" in df.columns else F.lit(None).cast("double"))
    df = df.withColumn("dewpoint_c", fahrenheit_to_celsius_expr("dwpf") if "dwpf" in df.columns else F.lit(None).cast("double"))

    optional_numeric("relh", "relh_pct")
    optional_numeric("drct", "wind_dir_deg")
    optional_numeric("sknt", "wind_speed_kt")
    optional_numeric("gust", "wind_gust_kt")
    optional_numeric("vsby", "visibility_sm")

    df = df.withColumn("cloud_cover_1", F.col("skyc1") if "skyc1" in df.columns else F.lit(None).cast("string"))

    optional_numeric("skyl1", "cloud_base_1_ft")

    df = df.withColumn("wx_codes", F.col("wxcodes") if "wxcodes" in df.columns else F.lit(None).cast("string"))

    optional_numeric("alti", "altimeter_inhg")

    df = df.withColumn("source", F.lit("IEM"))

    canonical_cols = [
        "icao_code",
        "obs_time_utc",
        "fecha_mes",
        "lat",
        "lon",
        "elevation_m",
        "bronze_metar",
        "temp_c",
        "dewpoint_c",
        "relh_pct",
        "wind_dir_deg",
        "wind_speed_kt",
        "wind_gust_kt",
        "visibility_sm",
        "cloud_cover_1",
        "cloud_base_1_ft",
        "wx_codes",
        "altimeter_inhg",
        "source",
    ]

    return df.select(*canonical_cols)

In [50]:
def split_iem_bronze_to_silver_by_airport_spark(
    icao_codes: list[str],
    icaos_colombia_set: set[str],
    min_rows_normalized: int = 500,
) -> DataFrame:
    frames = []
    diagnostics = []

    icaos_colombia_clean = {
        str(icao).strip().upper()
        for icao in icaos_colombia_set
        if str(icao).strip()
    }

    for icao in sorted(icao_codes):
        icao = str(icao).strip().upper()

        if not icao:
            continue

        if icao not in icaos_colombia_clean:
            diagnostics.append({
                "icao_code": icao,
                "reason": "icao_no_colombiano_o_fuera_de_dimension",
                "n_rows_bronze_total": 0,
                "n_rows_normalized_total": 0,
                "silver_saved": False,
            })
            continue

        paths = get_airport_paths(icao)
        bronze_dir = paths["bronze_iem_metar_dir"]
        parquet_files = sorted(bronze_dir.glob("historical_*.parquet"))

        if not parquet_files:
            diagnostics.append({
                "icao_code": icao,
                "reason": "sin_archivo_bronze",
                "n_rows_bronze_total": 0,
                "n_rows_normalized_total": 0,
                "silver_saved": False,
            })
            continue

        n_rows_bronze_total = sum(max(contar_filas_parquet(path), 0) for path in parquet_files)

        try:
            df_bronze = read_spark_parquet(parquet_files)
            df_norm = normalize_iem_metar_spark(df_bronze, icaos_colombia_clean)
            n_rows_normalized_total = df_norm.count()

            if n_rows_normalized_total < min_rows_normalized:
                diagnostics.append({
                    "icao_code": icao,
                    "reason": f"normalizado_menor_a_{min_rows_normalized}",
                    "n_rows_bronze_total": int(n_rows_bronze_total),
                    "n_rows_normalized_total": int(n_rows_normalized_total),
                    "silver_saved": False,
                })
                continue

            write_spark_parquet_dataset(
                df_norm,
                paths["silver_iem_metar_dir"] / "iem_metar_normalized_min500.parquet",
            )

            frames.append(df_norm)

            diagnostics.append({
                "icao_code": icao,
                "reason": "ok",
                "n_rows_bronze_total": int(n_rows_bronze_total),
                "n_rows_normalized_total": int(n_rows_normalized_total),
                "silver_saved": True,
            })

        except Exception as exc:
            diagnostics.append({
                "icao_code": icao,
                "reason": f"error_{type(exc).__name__}: {exc}",
                "n_rows_bronze_total": int(n_rows_bronze_total),
                "n_rows_normalized_total": 0,
                "silver_saved": False,
            })
            print(f"[WARN] No se pudo procesar IEM para {icao}: {exc}")

    diagnostics_df = spark.createDataFrame(diagnostics)

    write_spark_parquet_dataset(
        diagnostics_df,
        SILVER_DIAGNOSTICS_DIR / "iem_airports_split_to_silver_min500_diagnostics.parquet",
    )

    if frames:
        df_union = frames[0]

        for frame in frames[1:]:
            df_union = df_union.unionByName(frame, allowMissingColumns=True)

        return df_union

    return spark.createDataFrame([], schema=T.StructType([]))

In [51]:
silver_iem = split_iem_bronze_to_silver_by_airport_spark(
    icao_codes=icaos_para_iem,
    icaos_colombia_set=icaos_colombia_set,
    min_rows_normalized=MIN_IEM_ROWS,
)

write_spark_parquet_dataset(
    silver_iem,
    SILVER_IEM_DIR / "iem_metar_historical_normalized_min500.parquet",
)

print("IEM normalizado min500:")
print("Filas:", silver_iem.count())
print("Aeropuertos:", silver_iem.select("icao_code").distinct().count() if silver_iem.columns else 0)

display(silver_iem.limit(10))

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/iem/by_airport/SKAR/metar/iem_metar_normalized_min500.parquet | filas=33,122 | columnas=19 | part-files=8


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/iem/by_airport/SKAS/metar/iem_metar_normalized_min500.parquet | filas=9,348 | columnas=19 | part-files=8


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/iem/by_airport/SKBG/metar/iem_metar_normalized_min500.parquet | filas=42,174 | columnas=19 | part-files=8


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/iem/by_airport/SKBO/metar/iem_metar_normalized_min500.parquet | filas=57,325 | columnas=19 | part-files=8


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/iem/by_airport/SKBQ/metar/iem_metar_normalized_min500.parquet | filas=54,417 | columnas=19 | part-files=8
[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/iem/by_airport/SKBS/metar/iem_metar_normalized_min500.parquet | filas=8,039 | columnas=19 | part-files=8


...

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/iem_airports_split_to_silver_min500_diagnostics.parquet | filas=46 | columnas=5 | part-files=8


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/iem/iem_metar_historical_normalized_min500.parquet | filas=1,182,458 | columnas=19 | part-files=368
IEM normalizado min500:


Filas: 1182458


Aeropuertos: 46


icao_code,obs_time_utc,fecha_mes,lat,lon,elevation_m,bronze_metar,temp_c,dewpoint_c,relh_pct,wind_dir_deg,wind_speed_kt,wind_gust_kt,visibility_sm,cloud_cover_1,cloud_base_1_ft,wx_codes,altimeter_inhg,source
SKAR,2023-04-08 13:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 081300Z 0000...,19.000000000000004,19.000000000000004,100.0,0.0,0.0,NULL,3.73,BKN,1000.0,VCFG,30.09,IEM
SKAR,2023-04-08 14:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 081400Z 1900...,21.0,20.0,94.01,190.0,3.0,NULL,4.97,BKN,1300.0,VCFG,30.09,IEM
SKAR,2023-04-08 15:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 081500Z 0000...,21.999999999999996,19.000000000000004,83.09,0.0,0.0,NULL,6.21,SCT,1800.0,NULL,30.09,IEM
SKAR,2023-04-08 16:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 081600Z 3500...,25.0,19.000000000000004,69.33,350.0,3.0,NULL,6.21,SCT,2200.0,NULL,30.06,IEM
SKAR,2023-04-08 17:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 081700Z 3300...,26.0,19.000000000000004,65.33,330.0,4.0,NULL,6.21,SCT,2300.0,NULL,30.03,IEM
SKAR,2023-04-08 18:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 081800Z 2500...,26.999999999999996,19.000000000000004,61.58,250.0,3.0,NULL,6.21,BKN,2300.0,NULL,30.0,IEM
SKAR,2023-04-08 19:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 081900Z VRB0...,26.999999999999996,18.000000000000004,57.84,NULL,3.0,NULL,6.21,BKN,2300.0,NULL,29.94,IEM
SKAR,2023-04-08 20:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 082000Z VRB0...,29.000000000000004,19.000000000000004,54.8,NULL,3.0,NULL,6.21,FEW,2000.0,NULL,29.91,IEM
SKAR,2023-04-08 21:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 082100Z 0000...,29.000000000000004,19.000000000000004,54.8,0.0,0.0,NULL,6.21,FEW,2300.0,NULL,29.88,IEM
SKAR,2023-04-08 22:00:00,2023-04-01,4.5,-75.7167,1219.0,SKAR 082200Z 2200...,29.000000000000004,19.000000000000004,54.8,220.0,4.0,NULL,6.21,FEW,2500.0,NULL,29.88,IEM


In [52]:
def agregar_iem_a_mensual_spark(silver_iem: DataFrame) -> DataFrame:
    if not silver_iem.columns or silver_iem.limit(1).count() == 0:
        return spark.createDataFrame([], schema=T.StructType([]))

    df = silver_iem

    texto_meteo = F.upper(
        F.concat_ws(
            " ",
            F.coalesce(F.col("wx_codes").cast("string"), F.lit("")),
            F.coalesce(F.col("bronze_metar").cast("string"), F.lit("")),
        )
    )

    df = (
        df
        .withColumn("texto_meteo", texto_meteo)
        .withColumn("hay_lluvia", F.col("texto_meteo").rlike(r"\b(?:RA|DZ|SHRA|VCSH)\b").cast("int"))
        .withColumn("hay_tormenta", F.col("texto_meteo").rlike(r"\b(?:TS|TSRA|VCTS)\b").cast("int"))
        .withColumn("hay_niebla", F.col("texto_meteo").rlike(r"\b(?:FG|BR|HZ)\b").cast("int"))
        .withColumn("baja_visibilidad", (F.col("visibility_sm") < 3).cast("int"))
        .withColumn("viento_fuerte", (F.col("wind_speed_kt") >= 20).cast("int"))
        .withColumn("hay_rafaga", F.col("wind_gust_kt").isNotNull().cast("int"))
        .withColumn(
            "ifr_aprox",
            ((F.col("visibility_sm") < 3) | (F.col("cloud_base_1_ft") < 1000)).cast("int"),
        )
    )

    weather_monthly = (
        df
        .groupBy("icao_code", "fecha_mes")
        .agg(
            F.count("obs_time_utc").alias("n_metar_mes"),
            F.avg("temp_c").alias("temp_media_c"),
            F.min("temp_c").alias("temp_min_c"),
            F.max("temp_c").alias("temp_max_c"),
            F.stddev_samp("temp_c").alias("temp_std_c"),
            F.avg("dewpoint_c").alias("dewpoint_media_c"),
            F.avg("relh_pct").alias("humedad_media_pct"),
            F.avg("wind_speed_kt").alias("viento_medio_kt"),
            F.max("wind_speed_kt").alias("viento_max_kt"),
            F.stddev_samp("wind_speed_kt").alias("viento_std_kt"),
            F.max("wind_gust_kt").alias("rafaga_max_kt"),
            F.avg("visibility_sm").alias("visibilidad_media_sm"),
            F.min("visibility_sm").alias("visibilidad_min_sm"),
            F.stddev_samp("visibility_sm").alias("visibilidad_std_sm"),
            F.min("cloud_base_1_ft").alias("cloud_base_min_ft"),
            F.avg("altimeter_inhg").alias("altimeter_media_inhg"),
            F.avg("hay_lluvia").alias("prop_lluvia"),
            F.avg("hay_tormenta").alias("prop_tormenta"),
            F.avg("hay_niebla").alias("prop_niebla"),
            F.avg("baja_visibilidad").alias("prop_baja_visibilidad"),
            F.avg("viento_fuerte").alias("prop_viento_fuerte"),
            F.avg("hay_rafaga").alias("prop_rafaga"),
            F.avg("ifr_aprox").alias("prop_ifr_aprox"),
        )
        .withColumn(
            "n_metar_esperados_mes",
            F.dayofmonth(F.last_day("fecha_mes")) * F.lit(24),
        )
        .withColumn(
            "cobertura_metar_ratio",
            safe_divide_expr("n_metar_mes", "n_metar_esperados_mes"),
        )
        .orderBy("icao_code", "fecha_mes")
    )

    return weather_monthly

In [53]:
weather_monthly = agregar_iem_a_mensual_spark(silver_iem)

write_spark_parquet_dataset(
    weather_monthly,
    SILVER_IEM_DIR / "weather_monthly_iem_min500.parquet",
)

print("Clima mensual IEM min500:")
print("Filas:", weather_monthly.count())
print("Aeropuertos:", weather_monthly.select("icao_code").distinct().count() if weather_monthly.columns else 0)

display(weather_monthly.limit(10))

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/iem/weather_monthly_iem_min500.parquet | filas=2,495 | columnas=27 | part-files=1
Clima mensual IEM min500:


Filas: 2495


Aeropuertos: 46


icao_code,fecha_mes,n_metar_mes,temp_media_c,temp_min_c,temp_max_c,temp_std_c,dewpoint_media_c,humedad_media_pct,viento_medio_kt,viento_max_kt,viento_std_kt,rafaga_max_kt,visibilidad_media_sm,visibilidad_min_sm,visibilidad_std_sm,cloud_base_min_ft,altimeter_media_inhg,prop_lluvia,prop_tormenta,prop_niebla,prop_baja_visibilidad,prop_viento_fuerte,prop_rafaga,prop_ifr_aprox,n_metar_esperados_mes,cobertura_metar_ratio
SKAR,2020-01-01,573,23.653846153846153,15.0,33.00000000000001,4.334010316828683,18.793706293706293,76.9764335664337,1.2,12.0,1.983765392331008,NULL,5.809544658493886,0.62,1.0573798149208826,600.0,30.03521815008726,0.1082024432809773,0.012216404886561954,0.0506108202443281,0.047285464098073555,0.0,0.0,0.04903677758318739,744,0.7701612903225806
SKAR,2020-02-01,508,23.823762376237624,16.0,34.0,4.35555539518966,18.883168316831682,76.7702772277228,1.2693069306930693,13.0,2.053536562987683,NULL,5.627658730158736,0.19,1.3089271685649784,300.0,30.02348425196852,0.13385826771653545,0.025590551181102362,0.09645669291338582,0.06349206349206349,0.0,0.0,0.06746031746031746,696,0.7298850574712644
SKAR,2020-03-01,525,23.274193548387096,15.0,30.0,3.243239955496354,17.99794661190965,74.21174537987689,1.5487077534791251,14.0,2.3236857057363314,NULL,5.668011583011595,0.12,1.1737884011598065,800.0,30.029105367793274,0.1180952380952381,0.017142857142857144,0.1180952380952381,0.059845559845559844,0.0,0.0,0.06640625,744,0.7056451612903226
SKAR,2020-04-01,277,24.25221238938053,18.000000000000004,31.0,3.1542732170856076,19.51769911504425,76.47553097345138,1.736842105263158,9.0,2.3636477132610865,NULL,5.641805054151642,0.19,1.2982877484596589,200.0,30.067268722466956,0.17328519855595667,0.036101083032490974,0.05776173285198556,0.05415162454873646,0.0,0.0,0.08303249097472924,720,0.38472222222222224
SKAR,2020-05-01,208,25.4126213592233,20.0,32.0,2.790163195214897,20.28640776699029,74.46888349514565,1.3349514563106797,8.0,1.9654185298892481,NULL,5.911730769230784,1.24,0.8861134182296057,800.0,30.071835748792292,0.125,0.004807692307692308,0.0625,0.028846153846153848,0.0,0.0,0.04326923076923077,744,0.27956989247311825
SKAR,2020-06-01,190,24.83157894736842,17.0,32.0,2.840197591974757,20.194736842105264,76.51421052631582,1.3368421052631578,10.0,1.9632093706832194,NULL,5.660526315789488,0.5,1.3127654629367262,600.0,30.078526315789507,0.1631578947368421,0.010526315789473684,0.05263157894736842,0.07368421052631578,0.0,0.0,0.08421052631578947,720,0.2638888888888889
SKAR,2020-07-01,194,24.642487046632123,18.000000000000004,30.0,2.669593061819863,19.797927461139896,75.54031088082908,1.6321243523316062,7.0,2.1199459862119028,NULL,5.824948453608262,0.99,0.9908782138259487,500.0,30.068298969072163,0.17010309278350516,0.005154639175257732,0.05670103092783505,0.041237113402061855,0.0,0.0,0.05154639175257732,744,0.260752688172043
SKAR,2020-08-01,199,24.384615384615383,18.000000000000004,31.0,2.8969890574061816,18.353846153846153,70.51969230769235,2.4690721649484537,8.0,2.159824442335127,NULL,5.724020618556714,1.24,1.1092756912319706,500.0,30.073333333333345,0.16080402010050251,0.005025125628140704,0.06532663316582915,0.05670103092783505,0.0,0.0,0.06735751295336788,744,0.2674731182795699
SKAR,2020-09-01,190,24.11111111111111,18.000000000000004,29.000000000000004,2.5439400947494337,18.174603174603174,70.47365079365079,2.455026455026455,10.0,2.2323393474251105,NULL,5.911375661375676,1.86,0.7904854498507593,100.0,30.08423280423281,0.18421052631578946,0.0,0.06842105263157895,0.021164021164021163,0.0,0.0,0.037037037037037035,720,0.2638888888888889
SKAR,2020-10-01,2,21.0,20.0,21.999999999999996,1.4142135623730925,18.0,83.62,3.0,6.0,4.242640687119285,NULL,4.97,3.73,1.7536248173426379,2000.0,30.055,0.5,0.0,0.0,0.0,0.0,0.0,0.0,744,0.002688172043010753


In [54]:
if weather_monthly.columns and weather_monthly.limit(1).count() > 0:
    resumen_iem_min500 = (
        weather_monthly
        .groupBy("icao_code")
        .agg(
            F.countDistinct("fecha_mes").alias("meses_con_iem"),
            F.sum("n_metar_mes").alias("metar_total"),
            F.avg("cobertura_metar_ratio").alias("cobertura_promedio"),
            F.min("cobertura_metar_ratio").alias("cobertura_minima"),
            F.max("cobertura_metar_ratio").alias("cobertura_maxima"),
        )
        .orderBy(F.desc("meses_con_iem"), F.desc("metar_total"))
    )
else:
    resumen_iem_min500 = spark.createDataFrame([], schema=T.StructType([]))

write_spark_parquet_dataset(
    resumen_iem_min500,
    SILVER_DIAGNOSTICS_DIR / "iem_min500_monthly_coverage_summary.parquet",
)

display(resumen_iem_min500.limit(50))

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/iem_min500_monthly_coverage_summary.parquet | filas=46 | columnas=6 | part-files=1


icao_code,meses_con_iem,metar_total,cobertura_promedio,cobertura_minima,cobertura_maxima
SKBO,72,57325,1.0896310712544603,1.0026881720430108,1.2271505376344085
SKCL,72,55675,1.0583637536710393,0.9973118279569892,1.1102150537634408
SKRG,72,55003,1.0455716552832561,0.9986559139784946,1.0994623655913978
SKBQ,72,54417,1.0343930777853307,0.9637096774193549,1.1736111111111112
SKSP,72,52737,1.0024126908106794,0.9354838709677419,1.0739247311827957
SKLT,72,52351,0.9953332906026291,0.7258064516129032,1.1152777777777778
SKCG,72,51460,0.9783788262065636,0.5470430107526881,1.0902777777777777
SKPE,72,44650,0.8488868329718695,0.3575268817204301,1.1083333333333334
SKCC,72,43280,0.8227166002225675,0.4637096774193548,1.0376344086021505
SKBG,72,42174,0.8016131668442106,0.48118279569892475,1.131720430107527


$### 13.1. LIMPIEZA DE CARPETAS SILVER IEM VACÍAS

In [55]:
def diagnosticar_silver_iem_airport_dirs_vacios(base_dir: Path) -> DataFrame:
    base_dir = Path(base_dir)
    rows = []

    if not base_dir.exists():
        rows.append({
            "airport_code": None,
            "airport_dir": None,
            "exists": False,
            "n_files_total": 0,
            "n_dirs_total": 0,
            "size_total_bytes": 0,
            "delete_candidate": False,
            "reason": "base_dir_no_existe",
        })
        return spark.createDataFrame(rows)

    airport_dirs = sorted([
        path
        for path in base_dir.iterdir()
        if path.is_dir() and path.name.upper().startswith("SK")
    ])

    for airport_dir in airport_dirs:
        files_inside = [path for path in airport_dir.rglob("*") if path.is_file()]
        dirs_inside = [path for path in airport_dir.rglob("*") if path.is_dir()]

        n_files_total = len(files_inside)
        n_dirs_total = len(dirs_inside)
        size_total_bytes = sum(path.stat().st_size for path in files_inside)

        delete_candidate = n_files_total == 0

        reason = (
            "sin_archivos_en_todo_el_airport_dir"
            if delete_candidate
            else "con_archivos_no_borrar"
        )

        rows.append({
            "airport_code": airport_dir.name,
            "airport_dir": str(airport_dir),
            "exists": True,
            "n_files_total": n_files_total,
            "n_dirs_total": n_dirs_total,
            "size_total_bytes": int(size_total_bytes),
            "delete_candidate": bool(delete_candidate),
            "reason": reason,
        })

    return spark.createDataFrame(rows)

In [56]:
df_silver_iem_airport_dirs_diagnostic = diagnosticar_silver_iem_airport_dirs_vacios(
    SILVER_IEM_BY_AIRPORT_DIR
)

df_silver_iem_airport_dirs_to_delete = (
    df_silver_iem_airport_dirs_diagnostic
    .filter(F.col("delete_candidate") == True)
)

write_spark_parquet_dataset(
    df_silver_iem_airport_dirs_diagnostic,
    SILVER_DIAGNOSTICS_DIR / "silver_iem_airport_dirs_diagnostic_before_delete.parquet",
)

write_spark_parquet_dataset(
    df_silver_iem_airport_dirs_to_delete,
    SILVER_DIAGNOSTICS_DIR / "silver_iem_airport_dirs_to_delete.parquet",
)

print("Carpetas SK* revisadas:", df_silver_iem_airport_dirs_diagnostic.count())
print("Carpetas SK* candidatas a borrar completas:", df_silver_iem_airport_dirs_to_delete.count())

display(
    df_silver_iem_airport_dirs_to_delete
    .select(
        "airport_code",
        "airport_dir",
        "n_files_total",
        "n_dirs_total",
        "size_total_bytes",
        "reason",
    )
)

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/silver_iem_airport_dirs_diagnostic_before_delete.parquet | filas=146 | columnas=8 | part-files=8


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/silver_iem_airport_dirs_to_delete.parquet | filas=100 | columnas=8 | part-files=8


Carpetas SK* revisadas: 146


Carpetas SK* candidatas a borrar completas: 100


airport_code,airport_dir,n_files_total,n_dirs_total,size_total_bytes,reason
SKAA,data_proyecto_aer...,0,1,0,sin_archivos_en_t...
SKAC,data_proyecto_aer...,0,1,0,sin_archivos_en_t...
SKAD,data_proyecto_aer...,0,1,0,sin_archivos_en_t...
SKAG,data_proyecto_aer...,0,1,0,sin_archivos_en_t...
SKAL,data_proyecto_aer...,0,1,0,sin_archivos_en_t...
SKAM,data_proyecto_aer...,0,1,0,sin_archivos_en_t...
SKAP,data_proyecto_aer...,0,1,0,sin_archivos_en_t...
SKAT,data_proyecto_aer...,0,1,0,sin_archivos_en_t...
SKBC,data_proyecto_aer...,0,1,0,sin_archivos_en_t...
SKBE,data_proyecto_aer...,0,1,0,sin_archivos_en_t...


In [57]:
delete_results = []

if DELETE_EMPTY_SILVER_IEM_AIRPORT_DIRS:
    folders_to_delete = [
        row["airport_dir"]
        for row in df_silver_iem_airport_dirs_to_delete
        .select("airport_dir")
        .collect()
        if row["airport_dir"]
    ]

    for folder_path in folders_to_delete:
        airport_dir = Path(folder_path)

        try:
            if not airport_dir.exists():
                delete_results.append({
                    "airport_dir": str(airport_dir),
                    "status": "no_existe",
                })
                continue

            files_inside = [
                path
                for path in airport_dir.rglob("*")
                if path.is_file()
            ]

            if files_inside:
                delete_results.append({
                    "airport_dir": str(airport_dir),
                    "status": "no_borrada_tiene_archivos",
                })
                continue

            shutil.rmtree(airport_dir)

            delete_results.append({
                "airport_dir": str(airport_dir),
                "status": "borrada_completa",
            })

        except Exception as exc:
            delete_results.append({
                "airport_dir": str(airport_dir),
                "status": f"error_{type(exc).__name__}: {exc}",
            })
else:
    print("Modo seguro: no se borró nada.")

if delete_results:
    df_delete_silver_iem_airport_dirs_results = spark.createDataFrame(delete_results)

    write_spark_parquet_dataset(
        df_delete_silver_iem_airport_dirs_results,
        SILVER_DIAGNOSTICS_DIR / "silver_iem_airport_dirs_delete_results.parquet",
    )

    display(df_delete_silver_iem_airport_dirs_results.orderBy("status", "airport_dir"))

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/silver_iem_airport_dirs_delete_results.parquet | filas=100 | columnas=2 | part-files=8


airport_dir,status
data_proyecto_aer...,borrada_completa
data_proyecto_aer...,borrada_completa
data_proyecto_aer...,borrada_completa
data_proyecto_aer...,borrada_completa
data_proyecto_aer...,borrada_completa
data_proyecto_aer...,borrada_completa
data_proyecto_aer...,borrada_completa
data_proyecto_aer...,borrada_completa
data_proyecto_aer...,borrada_completa
data_proyecto_aer...,borrada_completa


In [58]:
df_silver_iem_airport_dirs_diagnostic_after = diagnosticar_silver_iem_airport_dirs_vacios(
    SILVER_IEM_BY_AIRPORT_DIR
)

write_spark_parquet_dataset(
    df_silver_iem_airport_dirs_diagnostic_after,
    SILVER_DIAGNOSTICS_DIR / "silver_iem_airport_dirs_diagnostic_after_delete.parquet",
)

print("Carpetas SK* restantes:", df_silver_iem_airport_dirs_diagnostic_after.count())

display(
    df_silver_iem_airport_dirs_diagnostic_after
    .groupBy("reason")
    .agg(
        F.count("airport_code").alias("n_airports"),
        F.sum("n_files_total").alias("archivos"),
        F.sum("size_total_bytes").alias("size_total_bytes"),
    )
)

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/silver/diagnostics/silver_iem_airport_dirs_diagnostic_after_delete.parquet | filas=46 | columnas=8 | part-files=8


Carpetas SK* restantes: 46


reason,n_airports,archivos,size_total_bytes
con_archivos_no_b...,46,828,39558792


### 14. CONSTRUIR DATASET GOLD FINAL

In [59]:
icaos_modelo_iem = {
    row["icao_code"]
    for row in silver_iem.select("icao_code").distinct().collect()
    if row["icao_code"] is not None
}

operations_airport_month_gold = operations_airport_month.filter(
    F.col("icao_code").isin(list(icaos_modelo_iem))
)

od_airport_month_gold = od_airport_month.filter(
    F.col("icao_code").isin(list(icaos_modelo_iem))
)

weather_monthly_gold = weather_monthly.filter(
    F.col("icao_code").isin(list(icaos_modelo_iem))
)

dataset_final = (
    operations_airport_month_gold
    .join(
        od_airport_month_gold,
        on=["icao_code", "fecha_mes"],
        how="left",
    )
    .join(
        weather_monthly_gold,
        on=["icao_code", "fecha_mes"],
        how="left",
    )
)

In [60]:
airport_dim_small = dim_airports.dropDuplicates(["icao_code"])

keep_cols = [
    col
    for col in [
        "icao_code",
        "name",
        "type",
        "municipality",
        "region_name",
        "iata_code",
        "latitude_deg",
        "longitude_deg",
        "elevation_ft",
        "scheduled_service",
    ]
    if col in airport_dim_small.columns
]

dataset_final = dataset_final.join(
    airport_dim_small.select(*keep_cols),
    on="icao_code",
    how="left",
)

if "n_metar_mes" in dataset_final.columns:
    dataset_final = (
        dataset_final
        .withColumn("n_metar_mes", F.coalesce(F.col("n_metar_mes"), F.lit(0)))
        .withColumn("tiene_metar_mes", (F.col("n_metar_mes") > 0).cast("int"))
    )
else:
    dataset_final = (
        dataset_final
        .withColumn("n_metar_mes", F.lit(0))
        .withColumn("tiene_metar_mes", F.lit(0))
    )

In [61]:
od_fill_cols = [
    "pasajeros_salida",
    "pasajeros_llegada",
    "pasajeros_total",
    "carga_salida_kg",
    "carga_llegada_kg",
    "carga_total_kg",
    "n_destinos_total",
    "n_origenes_total",
    "n_empresas_salida",
    "n_empresas_llegada",
    "n_registros_salida",
    "n_registros_llegada",
    "pasajeros_internacional_total",
    "proporcion_pasajeros_internacional",
]

od_fill_cols = [col for col in od_fill_cols if col in dataset_final.columns]

if od_fill_cols:
    dataset_final = dataset_final.fillna(0, subset=od_fill_cols)

od_signal_cols = [col for col in od_fill_cols if col in dataset_final.columns]

if od_signal_cols:
    od_flags = [F.col(col).cast("double") > 0 for col in od_signal_cols]
    tiene_od_expr = od_flags[0]

    for expr in od_flags[1:]:
        tiene_od_expr = tiene_od_expr | expr

    dataset_final = dataset_final.withColumn("tiene_od_mes", tiene_od_expr.cast("int"))
else:
    dataset_final = dataset_final.withColumn("tiene_od_mes", F.lit(0))

dataset_final = dataset_final.withColumn(
    "sin_od_sin_metar_mes",
    ((F.col("tiene_od_mes") == 0) & (F.col("tiene_metar_mes") == 0)).cast("int"),
)

dataset_final = dataset_final.withColumn("fecha_mes", F.to_date("fecha_mes"))

window_airport = Window.partitionBy("icao_code").orderBy("fecha_mes")

dataset_final = (
    dataset_final
    .orderBy("icao_code", "fecha_mes")
    .withColumn(
        "target_operaciones_total_mes_siguiente",
        F.lead("operaciones_total", 1).over(window_airport),
    )
)

write_spark_parquet_dataset(
    dataset_final,
    GOLD_MONTHLY_DIR / "monthly_airport_dataset.parquet",
)

print("Dataset final:")
display(dataset_final.limit(10))
print((dataset_final.count(), len(dataset_final.columns)))
print("Aeropuertos únicos:", dataset_final.select("icao_code").distinct().count())

[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/gold/monthly_airport_dataset/monthly_airport_dataset.parquet | filas=3,300 | columnas=63 | part-files=1
Dataset final:


icao_code,fecha_mes,operaciones_llegada,operaciones_salida,TOQUE Y DESPEGUE,operaciones_otras,operaciones_total,pasajeros_salida,carga_salida_kg,n_destinos_total,n_empresas_salida,n_registros_salida,pasajeros_llegada,carga_llegada_kg,n_origenes_total,n_empresas_llegada,n_registros_llegada,pasajeros_total,carga_total_kg,pasajeros_salida_internacional,pasajeros_salida_nacional,pasajeros_llegada_internacional,pasajeros_llegada_nacional,pasajeros_internacional_total,proporcion_pasajeros_internacional,n_metar_mes,temp_media_c,temp_min_c,temp_max_c,temp_std_c,dewpoint_media_c,humedad_media_pct,viento_medio_kt,viento_max_kt,viento_std_kt,rafaga_max_kt,visibilidad_media_sm,visibilidad_min_sm,visibilidad_std_sm,cloud_base_min_ft,altimeter_media_inhg,prop_lluvia,prop_tormenta,prop_niebla,prop_baja_visibilidad,prop_viento_fuerte,prop_rafaga,prop_ifr_aprox,n_metar_esperados_mes,cobertura_metar_ratio,name,type,municipality,region_name,iata_code,latitude_deg,longitude_deg,elevation_ft,scheduled_service,tiene_metar_mes,tiene_od_mes,sin_od_sin_metar_mes,target_operaciones_total_mes_siguiente
SKAR,2020-01-01,408.0,409.0,0.0,0.0,817.0,25548.0,119.0,73,5,85,22854.0,0.0,71,5,85,48402.0,119.0,4448.0,21100.0,3880.0,18974.0,8328.0,0.17205900582620554,573,23.653846153846153,15.0,33.00000000000001,4.334010316828683,18.793706293706293,76.9764335664337,1.2,12.0,1.983765392331008,NULL,5.809544658493886,0.62,1.0573798149208826,600.0,30.03521815008726,0.1082024432809773,0.012216404886561954,0.0506108202443281,0.047285464098073555,0.0,0.0,0.04903677758318739,744,0.7701612903225806,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,744.0
SKAR,2020-02-01,372.0,372.0,0.0,0.0,744.0,22490.0,182.0,66,6,76,22022.0,130.0,71,6,82,44512.0,312.0,3206.0,19284.0,2930.0,19092.0,6136.0,0.1378504672897196,508,23.823762376237624,16.0,34.0,4.35555539518966,18.883168316831682,76.7702772277228,1.2693069306930693,13.0,2.053536562987683,NULL,5.627658730158736,0.19,1.3089271685649784,300.0,30.02348425196852,0.13385826771653545,0.025590551181102362,0.09645669291338582,0.06349206349206349,0.0,0.0,0.06746031746031746,696,0.7298850574712644,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,531.0
SKAR,2020-03-01,265.0,266.0,0.0,0.0,531.0,13541.0,4170.0,69,7,82,13241.0,3660.0,67,7,87,26782.0,7830.0,1975.0,11566.0,1580.0,11661.0,3555.0,0.13273840639235307,525,23.274193548387096,15.0,30.0,3.243239955496354,17.99794661190965,74.21174537987689,1.5487077534791251,14.0,2.3236857057363314,NULL,5.668011583011595,0.12,1.1737884011598065,800.0,30.029105367793274,0.1180952380952381,0.017142857142857144,0.1180952380952381,0.059845559845559844,0.0,0.0,0.06640625,744,0.7056451612903226,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,21.0
SKAR,2020-04-01,11.0,10.0,0.0,0.0,21.0,0.0,17216.0,3,1,3,0.0,16709.0,2,1,2,0.0,33925.0,0.0,0.0,0.0,0.0,0.0,0.0,277,24.25221238938053,18.000000000000004,31.0,3.1542732170856076,19.51769911504425,76.47553097345138,1.736842105263158,9.0,2.3636477132610865,NULL,5.641805054151642,0.19,1.2982877484596589,200.0,30.067268722466956,0.17328519855595667,0.036101083032490974,0.05776173285198556,0.05415162454873646,0.0,0.0,0.08303249097472924,720,0.38472222222222224,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,31.0
SKAR,2020-05-01,18.0,13.0,0.0,0.0,31.0,0.0,7891.0,2,2,2,0.0,6061.0,2,2,2,0.0,13952.0,0.0,0.0,0.0,0.0,0.0,0.0,208,25.4126213592233,20.0,32.0,2.790163195214897,20.28640776699029,74.46888349514565,1.3349514563106797,8.0,1.9654185298892481,NULL,5.911730769230784,1.24,0.8861134182296057,800.0,30.071835748792292,0.125,0.004807692307692308,0.0625,0.028846153846153848,0.0,0.0,0.04326923076923077,744,0.27956989247311825,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,12.0
SKAR,2020-06-01,5.0,7.0,0.0,0.0,12.0,0.0,4275.0,2,2,2,1.0,10599.0,2,2,2,1.0,14874.0,0.0,0.0,

(3300, 63)


Aeropuertos únicos: 46


### 15. CREAR TARGET BAJO / MEDIO / ALTO

In [62]:
def clasificar_bajo_medio_alto_por_percentiles_spark(df: DataFrame) -> DataFrame:
    target_col = "target_operaciones_total_mes_siguiente"
    type_col = "type"

    df_work = (
        df
        .withColumn("_type_fill", F.coalesce(F.col(type_col).cast("string"), F.lit("desconocido")))
        .withColumn("_target_log1p", F.log1p(F.col(target_col).cast("double")))
    )

    stats = (
        df_work
        .filter(F.col(target_col).isNotNull())
        .groupBy("_type_fill")
        .agg(
            F.count(target_col).alias("_n_valid_target"),
            F.expr("percentile_approx(_target_log1p, array(0.33, 0.66), 10000)").alias("_q_log"),
        )
        .withColumn("target_q33_operaciones", F.expm1(F.col("_q_log")[0]))
        .withColumn("target_q66_operaciones", F.expm1(F.col("_q_log")[1]))
        .select(
            "_type_fill",
            "_n_valid_target",
            "target_q33_operaciones",
            "target_q66_operaciones",
        )
    )

    df_out = df_work.join(stats, on="_type_fill", how="left")

    df_out = df_out.withColumn(
        "target_nivel_operacion_mes_siguiente",
        F.when(F.col(target_col).isNull(), F.lit(None).cast("string"))
        .when(F.col("_n_valid_target") < 3, F.lit("bajo"))
        .when(F.col(target_col) <= F.col("target_q33_operaciones"), F.lit("bajo"))
        .when(F.col(target_col) <= F.col("target_q66_operaciones"), F.lit("medio"))
        .otherwise(F.lit("alto")),
    )

    df_out = (
        df_out
        .drop("_type_fill", "_target_log1p", "_n_valid_target")
        .withColumn("target_q33_operaciones", F.round("target_q33_operaciones", 1))
        .withColumn("target_q66_operaciones", F.round("target_q66_operaciones", 1))
    )

    print("── Umbrales por tipo de aeropuerto ──")
    display(stats.orderBy("_type_fill"))

    print("── Distribución global resultante ──")
    display(
        df_out
        .groupBy("target_nivel_operacion_mes_siguiente")
        .count()
        .orderBy("target_nivel_operacion_mes_siguiente")
    )

    return df_out

In [63]:
# ------------------------------------------------------------
# Limpiar columnas de target si ya existían por una ejecución previa
# ------------------------------------------------------------

cols_target_generadas = [
    "target_nivel_operacion_mes_siguiente",
    "target_q33_operaciones",
    "target_q66_operaciones",
]

for col_name in cols_target_generadas:
    if col_name in dataset_final.columns:
        dataset_final = dataset_final.drop(col_name)

print("Columnas target previas eliminadas si existían.")

# ------------------------------------------------------------
# Crear target bajo / medio / alto
# ------------------------------------------------------------

dataset_final = clasificar_bajo_medio_alto_por_percentiles_spark(dataset_final)

# Verificación defensiva de columnas duplicadas
cols = dataset_final.columns
duplicadas = sorted({
    col for col in cols
    if cols.count(col) > 1
})

if duplicadas:
    raise ValueError(f"Columnas duplicadas después de clasificar target: {duplicadas}")

dataset_modelo = dataset_final.filter(
    F.col("target_nivel_operacion_mes_siguiente").isNotNull()
)

write_spark_parquet_dataset(
    dataset_modelo,
    GOLD_MONTHLY_DIR / "monthly_airport_model_dataset.parquet",
)

print("Distribución del target:")

target_counts = (
    dataset_modelo
    .groupBy("target_nivel_operacion_mes_siguiente")
    .count()
)

total_df = target_counts.agg(
    F.sum("count").alias("total")
)

display(
    target_counts
    .crossJoin(total_df)
    .withColumn(
        "pct",
        F.col("count") / F.col("total") * F.lit(100.0)
    )
    .drop("total")
    .orderBy("target_nivel_operacion_mes_siguiente")
)

display(dataset_modelo.limit(10))

Columnas target previas eliminadas si existían.
── Umbrales por tipo de aeropuerto ──


_type_fill,_n_valid_target,target_q33_operaciones,target_q66_operaciones
large_airport,426,2564.0,4971.000000000003
medium_airport,2686,219.99999999999991,564.0000000000002
small_airport,142,412.9999999999999,539.0000000000002


── Distribución global resultante ──


target_nivel_operacion_mes_siguiente,count
NULL,46
alto,1105
bajo,1072
medio,1077


[OK] Dataset Parquet Spark guardado: data_proyecto_aeropuertos/gold/monthly_airport_dataset/monthly_airport_model_dataset.parquet | filas=3,254 | columnas=66 | part-files=1
Distribución del target:


target_nivel_operacion_mes_siguiente,count,pct
alto,1105,33.95820528580209
bajo,1072,32.9440688383528
medio,1077,33.097725875845114


icao_code,fecha_mes,operaciones_llegada,operaciones_salida,TOQUE Y DESPEGUE,operaciones_otras,operaciones_total,pasajeros_salida,carga_salida_kg,n_destinos_total,n_empresas_salida,n_registros_salida,pasajeros_llegada,carga_llegada_kg,n_origenes_total,n_empresas_llegada,n_registros_llegada,pasajeros_total,carga_total_kg,pasajeros_salida_internacional,pasajeros_salida_nacional,pasajeros_llegada_internacional,pasajeros_llegada_nacional,pasajeros_internacional_total,proporcion_pasajeros_internacional,n_metar_mes,temp_media_c,temp_min_c,temp_max_c,temp_std_c,dewpoint_media_c,humedad_media_pct,viento_medio_kt,viento_max_kt,viento_std_kt,rafaga_max_kt,visibilidad_media_sm,visibilidad_min_sm,visibilidad_std_sm,cloud_base_min_ft,altimeter_media_inhg,prop_lluvia,prop_tormenta,prop_niebla,prop_baja_visibilidad,prop_viento_fuerte,prop_rafaga,prop_ifr_aprox,n_metar_esperados_mes,cobertura_metar_ratio,name,type,municipality,region_name,iata_code,latitude_deg,longitude_deg,elevation_ft,scheduled_service,tiene_metar_mes,tiene_od_mes,sin_od_sin_metar_mes,target_operaciones_total_mes_siguiente,target_q33_operaciones,target_q66_operaciones,target_nivel_operacion_mes_siguiente
SKAR,2020-01-01,408.0,409.0,0.0,0.0,817.0,25548.0,119.0,73,5,85,22854.0,0.0,71,5,85,48402.0,119.0,4448.0,21100.0,3880.0,18974.0,8328.0,0.17205900582620554,573,23.653846153846153,15.0,33.00000000000001,4.334010316828683,18.793706293706293,76.9764335664337,1.2,12.0,1.983765392331008,NULL,5.809544658493886,0.62,1.0573798149208826,600.0,30.03521815008726,0.1082024432809773,0.012216404886561954,0.0506108202443281,0.047285464098073555,0.0,0.0,0.04903677758318739,744,0.7701612903225806,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,744.0,220.0,564.0,alto
SKAR,2020-02-01,372.0,372.0,0.0,0.0,744.0,22490.0,182.0,66,6,76,22022.0,130.0,71,6,82,44512.0,312.0,3206.0,19284.0,2930.0,19092.0,6136.0,0.1378504672897196,508,23.823762376237624,16.0,34.0,4.35555539518966,18.883168316831682,76.7702772277228,1.2693069306930693,13.0,2.053536562987683,NULL,5.627658730158736,0.19,1.3089271685649784,300.0,30.02348425196852,0.13385826771653545,0.025590551181102362,0.09645669291338582,0.06349206349206349,0.0,0.0,0.06746031746031746,696,0.7298850574712644,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,531.0,220.0,564.0,medio
SKAR,2020-03-01,265.0,266.0,0.0,0.0,531.0,13541.0,4170.0,69,7,82,13241.0,3660.0,67,7,87,26782.0,7830.0,1975.0,11566.0,1580.0,11661.0,3555.0,0.13273840639235307,525,23.274193548387096,15.0,30.0,3.243239955496354,17.99794661190965,74.21174537987689,1.5487077534791251,14.0,2.3236857057363314,NULL,5.668011583011595,0.12,1.1737884011598065,800.0,30.029105367793274,0.1180952380952381,0.017142857142857144,0.1180952380952381,0.059845559845559844,0.0,0.0,0.06640625,744,0.7056451612903226,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,21.0,220.0,564.0,bajo
SKAR,2020-04-01,11.0,10.0,0.0,0.0,21.0,0.0,17216.0,3,1,3,0.0,16709.0,2,1,2,0.0,33925.0,0.0,0.0,0.0,0.0,0.0,0.0,277,24.25221238938053,18.000000000000004,31.0,3.1542732170856076,19.51769911504425,76.47553097345138,1.736842105263158,9.0,2.3636477132610865,NULL,5.641805054151642,0.19,1.2982877484596589,200.0,30.067268722466956,0.17328519855595667,0.036101083032490974,0.05776173285198556,0.05415162454873646,0.0,0.0,0.08303249097472924,720,0.38472222222222224,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,31.0,220.0,564.0,bajo
SKAR,2020-05-01,18.0,13.0,0.0,0.0,31.0,0.0,7891.0,2,2,2,0.0,6061.0,2,2,2,0.0,13952.0,0.0,0.0,0.0,0.0,0.0,0.0,208,25.4126213592233,20.0,32.0,2.790163195214897,20.28640776699029,74.46888349514565,1.3349514563106797,8.0,1.9654185298892481,NULL,5.911730769230784,1.24,0.8861134182296057,800.0,30.071835748792292,0.125,0.004807692307692308,0.0625,0.028846153846153848,0.0,0.0,0.04326923076923077,744,0.27956989247311825,El Eden Airport,medium_airport,Armeni

### 16. DIAGNÓSTICOS DE COBERTURA

In [64]:
resumen_final = {
    "filas_dim_airports": dim_airports.count(),
    "filas_trafico_od_bronze": df_trafico_od.count(),
    "filas_operaciones_bronze": df_operaciones.count(),
    "filas_operations_airport_month": operations_airport_month.count(),
    "filas_od_airport_month": od_airport_month.count(),
    "filas_silver_iem": silver_iem.count(),
    "filas_weather_monthly": weather_monthly.count(),
    "filas_dataset_final": dataset_final.count(),
    "filas_dataset_modelo": dataset_modelo.count(),
    "aeropuertos_dataset_final": dataset_final.select("icao_code").distinct().count(),
    "aeropuertos_dataset_modelo": dataset_modelo.select("icao_code").distinct().count(),
}

save_json_to_file(
    resumen_final,
    GOLD_MONTHLY_DIR / "resumen_final.json",
)

print(json.dumps(resumen_final, indent=2, ensure_ascii=False))

[OK] JSON guardado: data_proyecto_aeropuertos/gold/monthly_airport_dataset/resumen_final.json
{
  "filas_dim_airports": 146,
  "filas_trafico_od_bronze": 455787,
  "filas_operaciones_bronze": 550724,
  "filas_operations_airport_month": 8673,
  "filas_od_airport_month": 5898,
  "filas_silver_iem": 1182458,
  "filas_weather_monthly": 2495,
  "filas_dataset_final": 3300,
  "filas_dataset_modelo": 3254,
  "aeropuertos_dataset_final": 46,
  "aeropuertos_dataset_modelo": 46
}


### 17. INSPECCIÓN FINAL DE ARCHIVOS GENERADOS

In [65]:
print("----- Silver Airports Colombia -----")
display(read_spark_parquet(SILVER_AIRPORTS_DIR / "dim_airports_colombia_skxx.parquet").limit(10))

print("----- Silver Operaciones Mensuales -----")
ops_month_parquet = SILVER_OPERACIONES_DIR / "operations_airport_month.parquet"
if ops_month_parquet.exists():
    display(read_spark_parquet(ops_month_parquet).limit(10))

print("----- Silver OD Mensual -----")
od_month_parquet = SILVER_TRAFICO_OD_DIR / "od_airport_month.parquet"
if od_month_parquet.exists():
    display(read_spark_parquet(od_month_parquet).limit(10))

print("----- Weather Monthly IEM Min500 -----")
weather_parquet = SILVER_IEM_DIR / "weather_monthly_iem_min500.parquet"
if weather_parquet.exists():
    display(read_spark_parquet(weather_parquet).limit(10))

print("----- Gold Monthly Dataset -----")
gold_parquet = GOLD_MONTHLY_DIR / "monthly_airport_dataset.parquet"
if gold_parquet.exists():
    display(read_spark_parquet(gold_parquet).limit(10))

print("----- Gold Model Dataset -----")
gold_model_parquet = GOLD_MONTHLY_DIR / "monthly_airport_model_dataset.parquet"
if gold_model_parquet.exists():
    display(read_spark_parquet(gold_model_parquet).limit(10))

print("Listo. Flujo PySpark completado.")

----- Silver Airports Colombia -----


id,ident,icao_code_original,gps_code,local_code,icao_code,icao_code_source_col,name,type,latitude_deg,longitude_deg,elevation_ft,iso_country,iso_region,region_name,municipality,scheduled_service,iata_code
40634,SK-054,,SKAA,PAZ,SKAA,gps_code,Caño Garza Airport,small_airport,5.591667,-71.589444,544,CO,CO-CAS,Casanare Department,Paz de Ariporo,0,
30615,SKAC,SKAC,SKAC,ACR,SKAC,icao_code,Araracuara Airport,small_airport,-0.600854,-72.398011,1250,CO,CO-CAQ,Caquetá Department,Araracuara,0,ACR
30613,SKAD,SKAD,SKAD,ACD,SKAD,icao_code,Alcides Fernández...,small_airport,8.497847,-77.274106,50,CO,CO-CHO,Chocó Department,Acandí,0,ACD
32306,SKAG,SKAG,SKAG,AGH,SKAG,icao_code,Hacaritama Airport,small_airport,8.247,-73.5814,545,CO,CO-CES,César Department,Aguachica,0,HAY
40796,SK-149,,SKAL,LLO,SKAL,gps_code,Calenturitas Airport,small_airport,9.652073,-73.495134,195,CO,CO-CES,César Department,La Loma,0,
30620,SKAM,SKAM,SKAM,AFI,SKAM,icao_code,Amalfi Airport,small_airport,6.895033,-75.047334,5507,CO,CO-ANT,Antioquía Department,Amalfi,0,AFI
6098,SKAP,SKAP,SKAP,APY,SKAP,icao_code,Gomez Nino Apiay ...,medium_airport,4.07607,-73.5627,1207,CO,CO-MET,Meta Department,Apiay,0,API
6099,SKAR,SKAR,SKAR,AXM,SKAR,icao_code,El Eden Airport,medium_airport,4.45278,-75.7664,3990,CO,CO-QUI,Quindio Department,Armenia,1,AXM
6100,SKAS,SKAS,SKAS,PUU,SKAS,icao_code,Tres De Mayo Airport,medium_airport,0.505228,-76.5008,815,CO,CO-PUT,Putumayo Department,Puerto Asís,1,PUU
429705,SKAT,SKAT,SKAT,ARQ,SKAT,icao_code,El Troncal Airport,small_airport,7.02106,-71.388901,512,CO,CO-ARA,Arauca Department,Arauquita,0,ARQ


----- Silver Operaciones Mensuales -----


icao_code,fecha_mes,operaciones_llegada,operaciones_salida,TOQUE Y DESPEGUE,operaciones_otras,operaciones_total
SKAC,2020-01-01,5.0,2.0,0.0,0.0,7.0
SKAC,2020-02-01,3.0,0.0,0.0,0.0,3.0
SKAC,2020-04-01,2.0,1.0,0.0,0.0,3.0
SKAC,2020-05-01,1.0,0.0,0.0,0.0,1.0
SKAC,2020-06-01,4.0,2.0,0.0,0.0,6.0
SKAC,2020-07-01,1.0,1.0,0.0,0.0,2.0
SKAC,2020-08-01,1.0,0.0,0.0,0.0,1.0
SKAC,2020-09-01,5.0,0.0,0.0,0.0,5.0
SKAC,2020-10-01,0.0,1.0,0.0,0.0,1.0
SKAC,2020-11-01,3.0,0.0,0.0,0.0,3.0


----- Silver OD Mensual -----


...

----- Gold Monthly Dataset -----


icao_code,fecha_mes,operaciones_llegada,operaciones_salida,TOQUE Y DESPEGUE,operaciones_otras,operaciones_total,pasajeros_salida,carga_salida_kg,n_destinos_total,n_empresas_salida,n_registros_salida,pasajeros_llegada,carga_llegada_kg,n_origenes_total,n_empresas_llegada,n_registros_llegada,pasajeros_total,carga_total_kg,pasajeros_salida_internacional,pasajeros_salida_nacional,pasajeros_llegada_internacional,pasajeros_llegada_nacional,pasajeros_internacional_total,proporcion_pasajeros_internacional,n_metar_mes,temp_media_c,temp_min_c,temp_max_c,temp_std_c,dewpoint_media_c,humedad_media_pct,viento_medio_kt,viento_max_kt,viento_std_kt,rafaga_max_kt,visibilidad_media_sm,visibilidad_min_sm,visibilidad_std_sm,cloud_base_min_ft,altimeter_media_inhg,prop_lluvia,prop_tormenta,prop_niebla,prop_baja_visibilidad,prop_viento_fuerte,prop_rafaga,prop_ifr_aprox,n_metar_esperados_mes,cobertura_metar_ratio,name,type,municipality,region_name,iata_code,latitude_deg,longitude_deg,elevation_ft,scheduled_service,tiene_metar_mes,tiene_od_mes,sin_od_sin_metar_mes,target_operaciones_total_mes_siguiente
SKAR,2020-01-01,408.0,409.0,0.0,0.0,817.0,25548.0,119.0,73,5,85,22854.0,0.0,71,5,85,48402.0,119.0,4448.0,21100.0,3880.0,18974.0,8328.0,0.17205900582620554,573,23.653846153846153,15.0,33.00000000000001,4.334010316828683,18.793706293706293,76.9764335664337,1.2,12.0,1.983765392331008,NULL,5.809544658493886,0.62,1.0573798149208826,600.0,30.03521815008726,0.1082024432809773,0.012216404886561954,0.0506108202443281,0.047285464098073555,0.0,0.0,0.04903677758318739,744,0.7701612903225806,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,744.0
SKAR,2020-02-01,372.0,372.0,0.0,0.0,744.0,22490.0,182.0,66,6,76,22022.0,130.0,71,6,82,44512.0,312.0,3206.0,19284.0,2930.0,19092.0,6136.0,0.1378504672897196,508,23.823762376237624,16.0,34.0,4.35555539518966,18.883168316831682,76.7702772277228,1.2693069306930693,13.0,2.053536562987683,NULL,5.627658730158736,0.19,1.3089271685649784,300.0,30.02348425196852,0.13385826771653545,0.025590551181102362,0.09645669291338582,0.06349206349206349,0.0,0.0,0.06746031746031746,696,0.7298850574712644,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,531.0
SKAR,2020-03-01,265.0,266.0,0.0,0.0,531.0,13541.0,4170.0,69,7,82,13241.0,3660.0,67,7,87,26782.0,7830.0,1975.0,11566.0,1580.0,11661.0,3555.0,0.13273840639235307,525,23.274193548387096,15.0,30.0,3.243239955496354,17.99794661190965,74.21174537987689,1.5487077534791251,14.0,2.3236857057363314,NULL,5.668011583011595,0.12,1.1737884011598065,800.0,30.029105367793274,0.1180952380952381,0.017142857142857144,0.1180952380952381,0.059845559845559844,0.0,0.0,0.06640625,744,0.7056451612903226,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,21.0
SKAR,2020-04-01,11.0,10.0,0.0,0.0,21.0,0.0,17216.0,3,1,3,0.0,16709.0,2,1,2,0.0,33925.0,0.0,0.0,0.0,0.0,0.0,0.0,277,24.25221238938053,18.000000000000004,31.0,3.1542732170856076,19.51769911504425,76.47553097345138,1.736842105263158,9.0,2.3636477132610865,NULL,5.641805054151642,0.19,1.2982877484596589,200.0,30.067268722466956,0.17328519855595667,0.036101083032490974,0.05776173285198556,0.05415162454873646,0.0,0.0,0.08303249097472924,720,0.38472222222222224,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,31.0
SKAR,2020-05-01,18.0,13.0,0.0,0.0,31.0,0.0,7891.0,2,2,2,0.0,6061.0,2,2,2,0.0,13952.0,0.0,0.0,0.0,0.0,0.0,0.0,208,25.4126213592233,20.0,32.0,2.790163195214897,20.28640776699029,74.46888349514565,1.3349514563106797,8.0,1.9654185298892481,NULL,5.911730769230784,1.24,0.8861134182296057,800.0,30.071835748792292,0.125,0.004807692307692308,0.0625,0.028846153846153848,0.0,0.0,0.04326923076923077,744,0.27956989247311825,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,12.0
SKAR,2020-06-01,5.0,7.0,0.0,0.0,12.0,0.0,4275.0,2,2,2,1.0,10599.0,2,2,2,1.0,14874.0,0.0,0.0,

----- Gold Model Dataset -----


icao_code,fecha_mes,operaciones_llegada,operaciones_salida,TOQUE Y DESPEGUE,operaciones_otras,operaciones_total,pasajeros_salida,carga_salida_kg,n_destinos_total,n_empresas_salida,n_registros_salida,pasajeros_llegada,carga_llegada_kg,n_origenes_total,n_empresas_llegada,n_registros_llegada,pasajeros_total,carga_total_kg,pasajeros_salida_internacional,pasajeros_salida_nacional,pasajeros_llegada_internacional,pasajeros_llegada_nacional,pasajeros_internacional_total,proporcion_pasajeros_internacional,n_metar_mes,temp_media_c,temp_min_c,temp_max_c,temp_std_c,dewpoint_media_c,humedad_media_pct,viento_medio_kt,viento_max_kt,viento_std_kt,rafaga_max_kt,visibilidad_media_sm,visibilidad_min_sm,visibilidad_std_sm,cloud_base_min_ft,altimeter_media_inhg,prop_lluvia,prop_tormenta,prop_niebla,prop_baja_visibilidad,prop_viento_fuerte,prop_rafaga,prop_ifr_aprox,n_metar_esperados_mes,cobertura_metar_ratio,name,type,municipality,region_name,iata_code,latitude_deg,longitude_deg,elevation_ft,scheduled_service,tiene_metar_mes,tiene_od_mes,sin_od_sin_metar_mes,target_operaciones_total_mes_siguiente,target_q33_operaciones,target_q66_operaciones,target_nivel_operacion_mes_siguiente
SKAR,2020-01-01,408.0,409.0,0.0,0.0,817.0,25548.0,119.0,73,5,85,22854.0,0.0,71,5,85,48402.0,119.0,4448.0,21100.0,3880.0,18974.0,8328.0,0.17205900582620554,573,23.653846153846153,15.0,33.00000000000001,4.334010316828683,18.793706293706293,76.9764335664337,1.2,12.0,1.983765392331008,NULL,5.809544658493886,0.62,1.0573798149208826,600.0,30.03521815008726,0.1082024432809773,0.012216404886561954,0.0506108202443281,0.047285464098073555,0.0,0.0,0.04903677758318739,744,0.7701612903225806,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,744.0,220.0,564.0,alto
SKAR,2020-02-01,372.0,372.0,0.0,0.0,744.0,22490.0,182.0,66,6,76,22022.0,130.0,71,6,82,44512.0,312.0,3206.0,19284.0,2930.0,19092.0,6136.0,0.1378504672897196,508,23.823762376237624,16.0,34.0,4.35555539518966,18.883168316831682,76.7702772277228,1.2693069306930693,13.0,2.053536562987683,NULL,5.627658730158736,0.19,1.3089271685649784,300.0,30.02348425196852,0.13385826771653545,0.025590551181102362,0.09645669291338582,0.06349206349206349,0.0,0.0,0.06746031746031746,696,0.7298850574712644,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,531.0,220.0,564.0,medio
SKAR,2020-03-01,265.0,266.0,0.0,0.0,531.0,13541.0,4170.0,69,7,82,13241.0,3660.0,67,7,87,26782.0,7830.0,1975.0,11566.0,1580.0,11661.0,3555.0,0.13273840639235307,525,23.274193548387096,15.0,30.0,3.243239955496354,17.99794661190965,74.21174537987689,1.5487077534791251,14.0,2.3236857057363314,NULL,5.668011583011595,0.12,1.1737884011598065,800.0,30.029105367793274,0.1180952380952381,0.017142857142857144,0.1180952380952381,0.059845559845559844,0.0,0.0,0.06640625,744,0.7056451612903226,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,21.0,220.0,564.0,bajo
SKAR,2020-04-01,11.0,10.0,0.0,0.0,21.0,0.0,17216.0,3,1,3,0.0,16709.0,2,1,2,0.0,33925.0,0.0,0.0,0.0,0.0,0.0,0.0,277,24.25221238938053,18.000000000000004,31.0,3.1542732170856076,19.51769911504425,76.47553097345138,1.736842105263158,9.0,2.3636477132610865,NULL,5.641805054151642,0.19,1.2982877484596589,200.0,30.067268722466956,0.17328519855595667,0.036101083032490974,0.05776173285198556,0.05415162454873646,0.0,0.0,0.08303249097472924,720,0.38472222222222224,El Eden Airport,medium_airport,Armenia,Quindio Department,AXM,4.45278,-75.7664,3990,1,1,1,0,31.0,220.0,564.0,bajo
SKAR,2020-05-01,18.0,13.0,0.0,0.0,31.0,0.0,7891.0,2,2,2,0.0,6061.0,2,2,2,0.0,13952.0,0.0,0.0,0.0,0.0,0.0,0.0,208,25.4126213592233,20.0,32.0,2.790163195214897,20.28640776699029,74.46888349514565,1.3349514563106797,8.0,1.9654185298892481,NULL,5.911730769230784,1.24,0.8861134182296057,800.0,30.071835748792292,0.125,0.004807692307692308,0.0625,0.028846153846153848,0.0,0.0,0.04326923076923077,744,0.27956989247311825,El Eden Airport,medium_airport,Armeni

Listo. Flujo PySpark completado.
